# Chapter 3. Flow Matching from Scratch

This notebook is the first runnable Chapter 3 implementation of Conditional Flow Matching (CFM) from scratch. It focuses on the main CFM pipeline and core visual diagnostics.

Key boundaries for this chapter:

- CFM training is local velocity regression: sample endpoint pairs, sample interpolation time, regress a velocity target. Training does not roll out the learned ODE.
- EB training, sampling, and quantitative evaluation are all performed in the 20D PC state space.
- PHATE 2D is visualization only. It is never used for EB training, sampling state updates, MMD, or Sliced W2.
- The 2D toy section is a teaching sanity check only. It is not used as an EB result or biological conclusion.
- CNF baselines, FFJORD likelihoods, large ablations, and final polished concept diagrams are intentionally deferred.

## 0. Setup

This setup cell fixes random seeds, locates the project root from either the repository root or `notebooks`, creates Chapter 3 output directories, and sets a consistent plotting style.

`QUICK_MODE=True` is the default first-version setting. For CI or smoke testing, set `CH03_SMOKE_MODE=1` before executing the notebook; that keeps the same code path but uses smaller data and fewer optimization steps.

In [1]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

CWD = Path.cwd().resolve()
root_candidates = [
    CWD,
    CWD.parent,
    CWD.parent.parent,
    CWD / "flow_matching_for_dynamic_biology",
    CWD.parent / "flow_matching_for_dynamic_biology",
    CWD.parent.parent / "flow_matching_for_dynamic_biology",
]
for candidate in root_candidates:
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError("Could not locate project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_seed, ensure_dir, savefig, save_table
from src.data import load_eb_timecourse_for_ch03, copy_trajectorynet_eb_to_data
from src.models import VelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample, midpoint_sample, odeint_sample
from src.metrics import endpoint_metrics, mmd_rbf, sliced_wasserstein_distance, trajectory_straightness
from src.flow_matching_reporting import plot_nfe_vs_error

SEED = 42
set_seed(SEED)
rng = np.random.default_rng(SEED)
FULL_MODE_RUN_START_PERF = time.perf_counter()
FULL_MODE_RUN_TIMESTAMP = pd.Timestamp.now(tz="Asia/Hong_Kong").isoformat()

QUICK_MODE = False
SMOKE_MODE = os.environ.get("CH03_SMOKE_MODE", "0") == "1"
PAPER_FIGURE_MODE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FULL_RUN_HINTS = {
    "eb_train_steps": 10000,
    "cnf_steps": 600,
    "time_ablation_steps": 1200,
    "capacity_steps": 1200,
    "notes": "Set QUICK_MODE=False manually for full-run settings; default remains quick and reproducible.",
}

FIG_DIR = ensure_dir(PROJECT_ROOT /  "figures" / "ch03")
TABLE_DIR = ensure_dir(PROJECT_ROOT /  "tables" / "ch03")
OUT_DIR = ensure_dir(PROJECT_ROOT /  "outputs" / "ch03")

PAPER_COLORS = {
    "source": "#4C78A8",
    "target": "#B8B8B8",
    "target_red": "#C44E52",
    "generated": "#2F7F73",
    "cfm": "#2F7FBD",
    "cnf": "#D55E00",
    "euler": "#4C78A8",
    "midpoint": "#55A868",
    "dopri5": "#C44E52",
    "low": "#2F7F73",
    "high": "#C44E52",
}


def set_paper_style(base_font_size=8.5):
    sns.set_theme(
        context="paper",
        style="white",
        font="DejaVu Sans",
        rc={
            "font.family": "DejaVu Sans",
            "font.size": base_font_size,
            "axes.titlesize": base_font_size + 0.8,
            "axes.labelsize": base_font_size,
            "xtick.labelsize": base_font_size - 1,
            "ytick.labelsize": base_font_size - 1,
            "legend.fontsize": base_font_size - 1,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": False,
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
            "savefig.pad_inches": 0.02,
            "lines.linewidth": 1.4,
            "lines.markersize": 3.5,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        },
    )
    plt.rcParams["figure.constrained_layout.use"] = True


set_paper_style()

figures_written = []
paper_ready_png_written = []
paper_ready_pdf_written = []
tables_written = []
paper_tables_written = []
skipped_items = []


def _figure_paths_from_name(filename_or_stem):
    name = str(filename_or_stem)
    path = Path(name)
    if path.suffix:
        stem = path.stem
    else:
        stem = name
    return FIG_DIR / f"{stem}.png", FIG_DIR / f"{stem}.pdf", stem


def save_figure(fig, filename, dpi=300, write_pdf=None):
    png_path, pdf_path, _ = _figure_paths_from_name(filename)
    ensure_dir(png_path.parent)
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=0.025)
    if str(png_path.relative_to(PROJECT_ROOT)) not in figures_written:
        figures_written.append(str(png_path.relative_to(PROJECT_ROOT)))
    if write_pdf is None:
        write_pdf = PAPER_FIGURE_MODE
    if write_pdf:
        fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.025)
        rel_pdf = str(pdf_path.relative_to(PROJECT_ROOT))
        if rel_pdf not in paper_ready_pdf_written:
            paper_ready_pdf_written.append(rel_pdf)
    plt.close(fig)
    return png_path


def save_paper_figure(fig, stem, dpi=300):
    png_path, pdf_path, _ = _figure_paths_from_name(stem)
    ensure_dir(png_path.parent)
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=0.025)
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.025)
    rel_png = str(png_path.relative_to(PROJECT_ROOT))
    rel_pdf = str(pdf_path.relative_to(PROJECT_ROOT))
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    if rel_png not in paper_ready_png_written:
        paper_ready_png_written.append(rel_png)
    if rel_pdf not in paper_ready_pdf_written:
        paper_ready_pdf_written.append(rel_pdf)
    plt.close(fig)
    return png_path, pdf_path


def save_csv(frame, filename):
    path = save_table(pd.DataFrame(frame), TABLE_DIR / filename)
    rel = str(path.relative_to(PROJECT_ROOT))
    if rel not in tables_written:
        tables_written.append(rel)
    return path


def _simple_markdown_table(table, index=False):
    display = table.reset_index() if index else table.copy()
    display = display.astype(object).where(pd.notnull(display), "")
    headers = [str(col) for col in display.columns]
    rows = [[str(value) for value in row] for row in display.to_numpy()]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

def save_paper_table(frame, stem, index=False):
    table = pd.DataFrame(frame)
    csv_path = TABLE_DIR / f"{stem}.csv"
    tex_path = TABLE_DIR / f"{stem}.tex"
    md_path = TABLE_DIR / f"{stem}.md"
    ensure_dir(TABLE_DIR)
    table.to_csv(csv_path, index=index)
    latex = table.to_latex(index=index, escape=False, float_format=lambda x: f"{x:.4g}")
    tex_path.write_text(latex, encoding="utf-8")
    try:
        markdown = table.to_markdown(index=index)
    except ImportError:
        markdown = _simple_markdown_table(table, index=index)
    md_path.write_text(markdown, encoding="utf-8")
    for path in [csv_path, tex_path, md_path]:
        rel = str(path.relative_to(PROJECT_ROOT))
        if rel not in paper_tables_written:
            paper_tables_written.append(rel)
    return csv_path, tex_path, md_path


def add_panel_label(ax, label, x=-0.08, y=1.05):
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="top", fontsize=9.5, fontweight="bold", color="0.12")


def short_strategy_label(strategy):
    return {
        "uniform": "uniform",
        "logit_normal_sigma_0.5": "logit σ=.5",
        "logit_normal_sigma_1.0": "logit σ=1",
        "logit_normal_sigma_2.0": "logit σ=2",
        "beta_2_2": "beta(2,2)",
        "beta_0.5_0.5": "beta(.5,.5)",
        "cosine": "cosine",
    }.get(str(strategy), str(strategy))


def clean_spines(ax, grid_axis=None):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color="0.90", linewidth=0.55)


def format_metric_axis(ax, metric):
    labels = {
        "mmd_20d": "MMD to target (20D PCs)",
        "endpoint_mmd_20d": "Endpoint MMD (20D PCs)",
        "sliced_w2_20d": "Sliced W2 (20D PCs)",
        "val_mse_20d": "Val CFM MSE (20D PCs)",
        "train_mse_20d": "Train CFM MSE (20D PCs)",
        "straightness_ratio_20d": "Straightness ratio in 20D PCs",
    }
    ax.set_ylabel(labels.get(metric, metric.replace("_", " ")))
    clean_spines(ax, grid_axis="y")


def add_note(ax, text, loc="lower left"):
    xy = {
        "lower left": (0.02, 0.03, "left", "bottom"),
        "lower right": (0.98, 0.03, "right", "bottom"),
        "upper left": (0.02, 0.97, "left", "top"),
        "upper right": (0.98, 0.97, "right", "top"),
    }[loc]
    ax.text(
        xy[0], xy[1], text, transform=ax.transAxes, ha=xy[2], va=xy[3], fontsize=7.0, color="0.25",
        bbox={"facecolor": "white", "edgecolor": "0.82", "pad": 2.0, "alpha": 0.88},
    )


def as_np(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def subsample_idx(n, max_n=None, seed=42):
    if max_n is None or n <= max_n:
        return np.arange(n)
    local_rng = np.random.default_rng(seed)
    return np.sort(local_rng.choice(n, size=int(max_n), replace=False))


def robust_limits(*arrays, q_low=1.0, q_high=99.0, margin=0.08):
    chunks = []
    for arr in arrays:
        if arr is None:
            continue
        arr = np.asarray(arr, dtype=float)
        if arr.size == 0:
            continue
        chunks.append(arr.reshape(-1, arr.shape[-1])[:, :2])
    if not chunks:
        return (-1.0, 1.0), (-1.0, 1.0)
    X = np.vstack(chunks)
    X = X[np.isfinite(X).all(axis=1)]
    lo = np.percentile(X, q_low, axis=0)
    hi = np.percentile(X, q_high, axis=0)
    span = np.maximum(hi - lo, 1e-6)
    lo = lo - margin * span
    hi = hi + margin * span
    return (float(lo[0]), float(hi[0])), (float(lo[1]), float(hi[1]))


def format_axis(ax, xlim=None, ylim=None, xlabel="state 1", ylabel="state 2", title=None):
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

print({
    "project_root": str(PROJECT_ROOT),
    "device": DEVICE,
    "quick_mode": QUICK_MODE,
    "smoke_mode": SMOKE_MODE,
    "figure_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "paper_figure_mode": PAPER_FIGURE_MODE,
    "full_run_hints": FULL_RUN_HINTS,
})

{'project_root': '/import/home4/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology', 'device': 'cuda', 'quick_mode': False, 'smoke_mode': False, 'figure_dir': '/import/home4/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology/figures/ch03', 'table_dir': '/import/home4/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology/tables/ch03', 'paper_figure_mode': True, 'full_run_hints': {'eb_train_steps': 10000, 'cnf_steps': 600, 'time_ablation_steps': 1200, 'capacity_steps': 1200, 'notes': 'Set QUICK_MODE=False manually for full-run settings; default remains quick and reproducible.'}}


## 1. 2D Toy Sanity Check

The toy section is a teaching concept figure, not an EB result. It uses an 8-Gaussian source and a single Gaussian target to make the CFM mechanism visible in two dimensions.

For each random endpoint pair, CFM samples `t ~ U(0,1)`, constructs `x_t=(1-t)x0+t x1`, and regresses the conditional path velocity `u_t=x1-x0`. The learned network estimates an Eulerian marginal vector field after seeing many such local regression samples.

In [2]:
def make_eight_gaussians(n=1500, radius=2.0, noise=0.08, seed=42):
    local_rng = np.random.default_rng(seed)
    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    centers = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])
    component = local_rng.integers(0, 8, size=n)
    points = centers[component] + local_rng.normal(scale=noise, size=(n, 2))
    return points.astype(np.float32), component


def make_single_gaussian(n=1500, loc=(0.0, 0.0), scale=(0.42, 0.28), angle=0.35, seed=43):
    local_rng = np.random.default_rng(seed)
    raw = local_rng.normal(size=(n, 2)).astype(np.float32)
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]], dtype=np.float32)
    S = np.diag(np.asarray(scale, dtype=np.float32))
    points = raw @ S @ R.T + np.asarray(loc, dtype=np.float32)
    return points.astype(np.float32)


def make_random_pair_batch_fn(X0, X1, seed=42):
    local_rng = np.random.default_rng(seed)

    def pair_batch_fn(batch_size):
        idx0 = local_rng.integers(0, len(X0), size=int(batch_size))
        idx1 = local_rng.integers(0, len(X1), size=int(batch_size))
        return {"x0": X0[idx0].astype(np.float32), "x1": X1[idx1].astype(np.float32)}

    return pair_batch_fn


toy_n = 1500 if QUICK_MODE else 5000
if SMOKE_MODE:
    toy_n = 350

toy_X0, toy_components = make_eight_gaussians(n=toy_n, seed=SEED)
toy_X1 = make_single_gaussian(n=toy_n, seed=SEED + 1)
toy_pair_batch_fn = make_random_pair_batch_fn(toy_X0, toy_X1, seed=SEED + 2)

toy_steps = 550 if QUICK_MODE else 2200
if SMOKE_MODE:
    toy_steps = 35
toy_log_every = 25 if not SMOKE_MODE else 5

toy_model = VelocityMLP(x_dim=2, hidden_dim=128, hidden_layers=4).to(DEVICE)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)
toy_history = train_cfm_steps(
    toy_model,
    toy_pair_batch_fn,
    toy_optimizer,
    n_steps=toy_steps,
    batch_size=256 if not SMOKE_MODE else 96,
    device=DEVICE,
    log_every=toy_log_every,
)

fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(toy_history["step"], toy_history["loss"], color="#2F6B5E", linewidth=1.6)
ax.set_xlabel("optimization step")
ax.set_ylabel("CFM local MSE")
ax.set_title("Toy CFM training loss")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "fig_toy_loss.png")
toy_history.tail()

,method,step,loss,wall_time_sec,sec_per_step,nfe_train_per_step,batch_size
84,cfm_local_regression,2100,0.559381,3.398212,0.001509,1,256
85,cfm_local_regression,2125,0.683748,3.435974,0.001510,1,256
86,cfm_local_regression,2150,0.619486,3.473703,0.001509,1,256
87,cfm_local_regression,2175,0.698403,3.511432,0.001509,1,256
88,cfm_local_regression,2200,0.614115,3.549214,0.001511,1,256


In [3]:
toy_model.eval()
toy_sample_n = min(900 if QUICK_MODE else 1800, len(toy_X0))
if SMOKE_MODE:
    toy_sample_n = min(220, len(toy_X0))
toy_sample_idx = subsample_idx(len(toy_X0), toy_sample_n, seed=SEED + 3)
toy_x0_t = torch.as_tensor(toy_X0[toy_sample_idx], dtype=torch.float32, device=DEVICE)
toy_euler_steps = 100 if not SMOKE_MODE else 25
with torch.no_grad():
    toy_final_t, toy_traj_t, toy_nfe = euler_sample(toy_model, toy_x0_t, n_steps=toy_euler_steps, return_traj=True)
toy_traj = as_np(toy_traj_t)

toy_times = [0.0, 0.25, 0.5, 0.75, 1.0]
toy_step_idx = np.round(np.asarray(toy_times) * (toy_traj.shape[0] - 1)).astype(int)
xlim, ylim = robust_limits(toy_X0, toy_X1, toy_traj.reshape(-1, 2), margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(12.3, 2.65), sharex=True, sharey=True)
for ax, tau, step in zip(axes, toy_times, toy_step_idx):
    pts = toy_traj[step]
    ax.scatter(toy_X1[subsample_idx(len(toy_X1), 450, seed=61), 0], toy_X1[subsample_idx(len(toy_X1), 450, seed=61), 1], s=5, color="0.72", alpha=0.20, linewidths=0)
    ax.scatter(pts[:, 0], pts[:, 1], s=6, color="#2F6B5E", alpha=0.52, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="toy x1", ylabel="toy x2", title=f"t={tau:.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
fig.suptitle("Toy population evolution under Euler sampling")
save_figure(fig, "fig_toy_evolution.png")
print({"toy_euler_nfe": toy_nfe, "toy_traj_shape": toy_traj.shape})

{'toy_euler_nfe': 100, 'toy_traj_shape': (101, 1800, 2)}


### Figure 3.2: Conditional Velocity Versus Marginal Velocity

A single endpoint pair gives one conditional straight-line velocity. Many endpoint pairs passing through the same local region give many possible conditional velocities. The neural velocity field learns their local average behavior as a marginal vector field estimate.

In [4]:
def plot_toy_conditional_vs_marginal(model, X0, X1, device="cpu", seed=42):
    local_rng = np.random.default_rng(seed)
    n_pairs = min(1800, len(X0), len(X1))
    i0 = local_rng.integers(0, len(X0), size=n_pairs)
    i1 = local_rng.integers(0, len(X1), size=n_pairs)
    x0 = X0[i0]
    x1 = X1[i1]
    t_value = 0.5
    x_t = (1.0 - t_value) * x0 + t_value * x1
    u_t = x1 - x0

    center = np.median(x_t, axis=0)
    dist = np.linalg.norm(x_t - center[None, :], axis=1)
    radius = np.quantile(dist, 0.18)
    local = np.flatnonzero(dist <= max(radius, 1e-6))
    if len(local) > 120:
        local = local[subsample_idx(len(local), 120, seed=seed + 1)]

    model.eval()
    with torch.no_grad():
        center_t = torch.as_tensor(center[None, :], dtype=torch.float32, device=device)
        time_t = torch.full((1, 1), t_value, dtype=torch.float32, device=device)
        pred = model(center_t, time_t).detach().cpu().numpy()[0]
    mean_conditional = u_t[local].mean(axis=0)

    xlim, ylim = robust_limits(X0, X1, x_t[local], margin=0.12)
    fig, ax = plt.subplots(figsize=(5.2, 4.5))
    ax.scatter(X0[subsample_idx(len(X0), 550, seed=71), 0], X0[subsample_idx(len(X0), 550, seed=71), 1], s=5, color="#4C78A8", alpha=0.18, linewidths=0, label="source")
    ax.scatter(X1[subsample_idx(len(X1), 550, seed=72), 0], X1[subsample_idx(len(X1), 550, seed=72), 1], s=5, color="#D1495B", alpha=0.18, linewidths=0, label="target")
    ax.scatter(x_t[local, 0], x_t[local, 1], s=11, color="0.55", alpha=0.35, linewidths=0)
    ax.quiver(x_t[local, 0], x_t[local, 1], u_t[local, 0], u_t[local, 1], angles="xy", scale_units="xy", scale=12, width=0.003, color="#9CC9C2", alpha=0.52, label="conditional velocities")
    ax.quiver([center[0]], [center[1]], [mean_conditional[0]], [mean_conditional[1]], angles="xy", scale_units="xy", scale=5, width=0.010, color="0.18", alpha=0.85, label="local conditional average")
    ax.quiver([center[0]], [center[1]], [pred[0]], [pred[1]], angles="xy", scale_units="xy", scale=5, width=0.013, color="#B9352B", alpha=0.95, label="network prediction")
    ax.scatter([center[0]], [center[1]], s=42, color="#B9352B", edgecolor="white", linewidth=0.7, zorder=5)
    format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2", title="Conditional velocities collapse to a marginal vector")
    ax.legend(frameon=False, loc="best")
    return fig, {"center": center, "network_velocity": pred, "mean_conditional_velocity": mean_conditional}

fig, toy_fig32_info = plot_toy_conditional_vs_marginal(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 4)
save_figure(fig, "fig03_02_conditional_vs_marginal_toy.png")
toy_fig32_info

{'center': array([-0.01860185, -0.01249047], dtype=float32),
 'network_velocity': array([0.49289185, 0.44987357], dtype=float32),
 'mean_conditional_velocity': array([ 0.39620182, -0.12577431], dtype=float32)}

### Figure 3.3: CFM Object Hierarchy

This figure separates the objects in CFM: one endpoint path, a population of endpoint-conditioned paths, and the learned marginal vector field at `t=0.5`. It is still a 2D teaching toy, not the EB analysis state space.

In [5]:
def plot_toy_cfm_object_hierarchy(model, X0, X1, device="cpu", seed=42):
    local_rng = np.random.default_rng(seed)
    n_paths = min(90, len(X0), len(X1))
    i0 = local_rng.integers(0, len(X0), size=n_paths)
    i1 = local_rng.integers(0, len(X1), size=n_paths)
    x0 = X0[i0]
    x1 = X1[i1]
    xlim, ylim = robust_limits(X0, X1, x0, x1, margin=0.12)

    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), sharex=True, sharey=True)
    for ax in axes:
        ax.scatter(X0[subsample_idx(len(X0), 500, seed=81), 0], X0[subsample_idx(len(X0), 500, seed=81), 1], s=5, color="#4C78A8", alpha=0.14, linewidths=0)
        ax.scatter(X1[subsample_idx(len(X1), 500, seed=82), 0], X1[subsample_idx(len(X1), 500, seed=82), 1], s=5, color="#D1495B", alpha=0.14, linewidths=0)
        format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2")

    a, b = x0[0], x1[0]
    path = np.stack([(1.0 - t) * a + t * b for t in np.linspace(0, 1, 50)], axis=0)
    axes[0].plot(path[:, 0], path[:, 1], color="0.18", linewidth=1.6)
    axes[0].scatter([a[0], b[0]], [a[1], b[1]], s=40, color=["#4C78A8", "#D1495B"], edgecolor="white", linewidth=0.7, zorder=4)
    mid = 0.5 * (a + b)
    vel = b - a
    axes[0].quiver([mid[0]], [mid[1]], [vel[0]], [vel[1]], angles="xy", scale_units="xy", scale=4, width=0.012, color="#2F6B5E")
    axes[0].set_title("A. one endpoint path")

    for j in range(n_paths):
        axes[1].plot([x0[j, 0], x1[j, 0]], [x0[j, 1], x1[j, 1]], color="0.25", alpha=0.18, linewidth=0.65)
    axes[1].set_title("B. many endpoint paths")
    axes[1].set_ylabel("")

    grid_n = 20
    xs = np.linspace(xlim[0], xlim[1], grid_n)
    ys = np.linspace(ylim[0], ylim[1], grid_n)
    gx, gy = np.meshgrid(xs, ys)
    pts = np.column_stack([gx.ravel(), gy.ravel()]).astype(np.float32)
    model.eval()
    with torch.no_grad():
        x_t = torch.as_tensor(pts, dtype=torch.float32, device=device)
        t_t = torch.full((pts.shape[0], 1), 0.5, dtype=torch.float32, device=device)
        v = model(x_t, t_t).detach().cpu().numpy()
    norm = np.linalg.norm(v, axis=1)
    cap = np.percentile(norm[norm > 0], 88) if np.any(norm > 0) else 1.0
    v = v * np.minimum(1.0, cap / np.clip(norm[:, None], 1e-12, None))
    axes[2].quiver(pts[:, 0], pts[:, 1], v[:, 0], v[:, 1], angles="xy", scale_units="xy", scale=16, width=0.004, color="#B9352B", alpha=0.72)
    axes[2].set_title("C. learned marginal field at t=0.5")
    axes[2].set_ylabel("")
    fig.suptitle("CFM object hierarchy on a 2D toy problem")
    return fig

fig = plot_toy_cfm_object_hierarchy(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 5)
save_figure(fig, "fig03_03_cfm_object_hierarchy_toy.png")

PosixPath('/import/home4/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology/figures/ch03/fig03_03_cfm_object_hierarchy_toy.png')

## 2. Load EB Data for 20D CFM

The EB main experiment uses `load_eb_timecourse_for_ch03`, which returns 20D PC coordinates as `X_cost` and 2D PHATE coordinates as `X_plot`.

The state used by the EB model is `X_cost[:, :20]`. PHATE 2D is visualization only and does not enter training, sampling, MMD, or Sliced W2. If the copied EB asset is missing, the notebook attempts to copy it into `data/trajectorynet_eb/`.

In [6]:
EB_PATH = PROJECT_ROOT / "data" / "trajectorynet_eb" / "eb_velocity_v5.npz"
EB_SOURCE_PATH = PROJECT_ROOT.parent / "baselines" / "trajectorynet" / "data" / "eb_velocity_v5.npz"
if not EB_PATH.exists():
    print(f"{EB_PATH} not found; attempting copy from {EB_SOURCE_PATH}")
    copy_trajectorynet_eb_to_data(source_path=EB_SOURCE_PATH, target_path=EB_PATH)

max_cells_per_time = 750 if QUICK_MODE else 1800
if SMOKE_MODE:
    max_cells_per_time = 90

eb = load_eb_timecourse_for_ch03(
    EB_PATH,
    cost_embedding="pcs",
    plot_embedding="phate",
    n_cost_dims=20,
    max_cells_per_time=max_cells_per_time,
    seed=SEED,
)

selected_counts = pd.Series(eb["time"], name="time").value_counts().sort_index().rename_axis("time").reset_index(name="selected_n_cells")
count_table = eb["full_counts_by_time"].merge(selected_counts, on="time", how="left")
count_table["selected_n_cells"] = count_table["selected_n_cells"].fillna(0).astype(int)
save_csv(count_table, "ch03_eb_timepoint_counts.csv")

time_labels = sorted(pd.Series(eb["time"]).unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))
if len(time_labels) < 2:
    raise ValueError(f"Need at least two EB time labels; found {time_labels}")
source_time = str(time_labels[0])
target_time = str(time_labels[1])
print(f"Selected adjacent EB time labels: source={source_time}, target={target_time}")
print(count_table[["time", "n_cells", "selected_n_cells"]].to_string(index=False))

mask_source = eb["time"].astype(str) == source_time
mask_target = eb["time"].astype(str) == target_time
X_source_20d = np.asarray(eb["X_cost"][mask_source], dtype=np.float32)
X_target_20d = np.asarray(eb["X_cost"][mask_target], dtype=np.float32)
X_source_phate = np.asarray(eb["X_plot"][mask_source], dtype=np.float32)
X_target_phate = np.asarray(eb["X_plot"][mask_target], dtype=np.float32)
cell_source = np.asarray(eb["cell_id"])[mask_source]
cell_target = np.asarray(eb["cell_id"])[mask_target]

assert X_source_20d.shape[1] == 20 and X_target_20d.shape[1] == 20
assert X_source_phate.shape[1] == 2 and X_target_phate.shape[1] == 2
print({
    "X_source_20d": X_source_20d.shape,
    "X_target_20d": X_target_20d.shape,
    "X_source_phate": X_source_phate.shape,
    "X_target_phate": X_target_phate.shape,
})

Selected adjacent EB time labels: source=0, target=1
time  n_cells  selected_n_cells
   0     2381              1800
   1     4163              1800
   2     3278              1800
   3     3665              1800
   4     3332              1800
{'X_source_20d': (1800, 20), 'X_target_20d': (1800, 20), 'X_source_phate': (1800, 2), 'X_target_phate': (1800, 2)}


## 3. EB Train/Val Split and Pairing

The split is by cells, not by generated endpoint pairs. Source cells are split 80/20, target cells are split 80/20, training pairs are sampled only from train source and train target cells, and validation pairs are sampled only from held-out source and target cells.

The validation MSE below uses a fixed held-out endpoint-pair set and a fixed interpolation-time grid. This makes the validation curve deterministic across logging events. All validation loss values are computed in 20D PC space.

In [7]:
def train_val_indices(n, train_fraction=0.8, seed=42):
    local_rng = np.random.default_rng(seed)
    perm = local_rng.permutation(n)
    n_train = int(np.floor(float(train_fraction) * n))
    n_train = min(max(n_train, 1), n - 1)
    train = np.sort(perm[:n_train])
    val = np.sort(perm[n_train:])
    return train, val

src_train_idx, src_val_idx = train_val_indices(len(X_source_20d), seed=SEED + 10)
tgt_train_idx, tgt_val_idx = train_val_indices(len(X_target_20d), seed=SEED + 11)

X0_train = X_source_20d[src_train_idx]
X1_train = X_target_20d[tgt_train_idx]
X0_val = X_source_20d[src_val_idx]
X1_val = X_target_20d[tgt_val_idx]
X0_train_phate = X_source_phate[src_train_idx]
X1_train_phate = X_target_phate[tgt_train_idx]
X0_val_phate = X_source_phate[src_val_idx]
X1_val_phate = X_target_phate[tgt_val_idx]

pair_rng = np.random.default_rng(SEED + 12)

def pair_batch_fn(batch_size):
    idx0 = pair_rng.integers(0, len(X0_train), size=int(batch_size))
    idx1 = pair_rng.integers(0, len(X1_train), size=int(batch_size))
    return {"x0": X0_train[idx0].astype(np.float32), "x1": X1_train[idx1].astype(np.float32)}

val_pair_n = 1200 if QUICK_MODE else 2500
if SMOKE_MODE:
    val_pair_n = 180
val_pair_rng = np.random.default_rng(SEED + 13)
val_i0 = val_pair_rng.integers(0, len(X0_val), size=int(val_pair_n))
val_i1 = val_pair_rng.integers(0, len(X1_val), size=int(val_pair_n))
val_x0 = X0_val[val_i0].astype(np.float32)
val_x1 = X1_val[val_i1].astype(np.float32)
val_t_grid = np.asarray([0.25, 0.50, 0.75], dtype=np.float32)

split_table = pd.DataFrame([
    {"population": "source", "time": source_time, "train_cells": len(X0_train), "val_cells": len(X0_val)},
    {"population": "target", "time": target_time, "train_cells": len(X1_train), "val_cells": len(X1_val)},
])
save_csv(split_table, "ch03_eb20d_train_val_split.csv")
split_table

,population,time,train_cells,val_cells
0,source,0,1440,360
1,target,1,1440,360


## 4. Train EB 20D VelocityMLP

The EB model is trained in the 20D PC state space using `VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4)`. The loss is CFM local velocity regression in 20D.

No PHATE coordinates appear in this training loop. PHATE is only used later for plotting generated samples.

In [8]:
def val_cfm_mse(model, x0_np, x1_np, t_values, device="cpu", max_eval_pairs=None):
    model.eval()
    if max_eval_pairs is not None and len(x0_np) > max_eval_pairs:
        idx = subsample_idx(len(x0_np), max_eval_pairs, seed=SEED + 14)
        x0_np = x0_np[idx]
        x1_np = x1_np[idx]
    x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=device)
    x1_t = torch.as_tensor(x1_np, dtype=torch.float32, device=device)
    losses = []
    with torch.no_grad():
        for t_value in t_values:
            t_t = torch.full((x0_t.shape[0], 1), float(t_value), dtype=torch.float32, device=device)
            loss = cfm_loss_from_pairs(model, x0_t, x1_t, s=t_t)
            losses.append(float(loss.detach().cpu()))
    model.train()
    return float(np.mean(losses))

set_seed(SEED + 20)
eb_model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
eb_optimizer = torch.optim.Adam(eb_model.parameters(), lr=1e-3)

eb_steps = 1500 if QUICK_MODE else 10000
if SMOKE_MODE:
    eb_steps = 45
eb_batch_size = 256
if SMOKE_MODE:
    eb_batch_size = 96
log_every = 100 if not SMOKE_MODE else 10

rows = []
eb_model.train()
start = time.perf_counter()
last_log_step = 0
last_log_time = start
for step in range(1, eb_steps + 1):
    batch = pair_batch_fn(eb_batch_size)
    x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
    x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
    loss = cfm_loss_from_pairs(eb_model, x0, x1)
    eb_optimizer.zero_grad()
    loss.backward()
    eb_optimizer.step()

    if step == 1 or step % log_every == 0 or step == eb_steps:
        now = time.perf_counter()
        val_mse = val_cfm_mse(eb_model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=1200 if not SMOKE_MODE else 180)
        rows.append({
            "step": int(step),
            "train_loss_20d": float(loss.detach().cpu()),
            "val_mse_20d_fixed_t_grid": float(val_mse),
            "wall_time_sec": float(now - start),
            "sec_per_step_since_last_log": float((now - last_log_time) / max(1, step - last_log_step)),
            "batch_size": int(eb_batch_size),
            "x_dim": 20,
            "hidden_dim": 256,
            "hidden_layers": 4,
            "n_parameters": count_parameters(eb_model),
            "device": DEVICE,
        })
        last_log_step = step
        last_log_time = now

eb_history = pd.DataFrame(rows)
save_csv(eb_history, "ch03_eb20d_training_log.csv")

fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.plot(eb_history["step"], eb_history["train_loss_20d"], marker="o", markersize=2.5, linewidth=1.4, color="#2F6B5E", label="train CFM MSE")
ax.plot(eb_history["step"], eb_history["val_mse_20d_fixed_t_grid"], marker="s", markersize=2.5, linewidth=1.4, color="#B9352B", label="val CFM MSE")
ax.set_xlabel("optimization step")
ax.set_ylabel("20D velocity-regression MSE")
ax.set_title("EB 20D CFM training and validation loss")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False)
save_figure(fig, "figB1_eb20d_train_val_loss.png")
eb_history.tail()

,step,train_loss_20d,val_mse_20d_fixed_t_grid,wall_time_sec,sec_per_step_since_last_log,batch_size,x_dim,hidden_dim,hidden_layers,n_parameters,device
96,9600,3.340216,5.126673,15.062417,0.001514,256,20,256,4,208148,cuda
97,9700,2.941583,5.092643,15.213119,0.001507,256,20,256,4,208148,cuda
98,9800,3.372888,5.114997,15.365369,0.001523,256,20,256,4,208148,cuda
99,9900,3.518093,5.153931,15.516835,0.001515,256,20,256,4,208148,cuda
100,10000,3.197348,5.149392,15.667371,0.001505,256,20,256,4,208148,cuda


## 5. EB Endpoint Pair Visualization in PHATE

Figure 3.4 draws sampled train endpoint pairs in PHATE 2D for interpretability. The CFM training paths are straight lines in 20D PC space. Their PHATE-projected line segments are visualization-only chords and do not represent true 20D geometry or observed biological trajectories.

In [9]:
def plot_endpoint_pairs_phate(X0_plot, X1_plot, n_pairs=80, seed=42):
    local_rng = np.random.default_rng(seed)
    n_pairs = min(int(n_pairs), max(len(X0_plot), len(X1_plot)))
    idx0 = local_rng.integers(0, len(X0_plot), size=n_pairs)
    idx1 = local_rng.integers(0, len(X1_plot), size=n_pairs)
    x0p = X0_plot[idx0]
    x1p = X1_plot[idx1]
    xlim, ylim = robust_limits(X0_plot, X1_plot, x0p, x1p, margin=0.10)
    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    ax.scatter(X0_plot[subsample_idx(len(X0_plot), 650, seed=91), 0], X0_plot[subsample_idx(len(X0_plot), 650, seed=91), 1], s=7, color="#3267A8", alpha=0.38, linewidths=0, label=f"source time {source_time}")
    ax.scatter(X1_plot[subsample_idx(len(X1_plot), 650, seed=92), 0], X1_plot[subsample_idx(len(X1_plot), 650, seed=92), 1], s=7, color="#C9463D", alpha=0.34, linewidths=0, label=f"target time {target_time}")
    for a, b in zip(x0p, x1p):
        ax.plot([a[0], b[0]], [a[1], b[1]], color="0.25", alpha=0.24, linewidth=0.55)
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title="Sampled EB train endpoint pairs displayed in PHATE")
    ax.legend(frameon=False, loc="best")
    ax.text(0.02, 0.02, "20D training chords; PHATE display only", transform=ax.transAxes, fontsize=7, color="0.25", bbox={"facecolor": "white", "edgecolor": "0.85", "pad": 2.0, "alpha": 0.92})
    return fig

pair_plot_n = 80 if QUICK_MODE else 120
if SMOKE_MODE:
    pair_plot_n = 35
fig = plot_endpoint_pairs_phate(X0_train_phate, X1_train_phate, n_pairs=pair_plot_n, seed=SEED + 30)
save_figure(fig, "fig03_04_eb_endpoint_pairs_phate.png")

PosixPath('/import/home4/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology/figures/ch03/fig03_04_eb_endpoint_pairs_phate.png')

## 6. EB Sampling and Population Evolution

Sampling integrates the learned velocity field in 20D PC space with `src.sampling.euler_sample`. The recorded generated states are then projected to PHATE-like coordinates for visualization.

The first-version projection is a kNN regressor fit from observed `X_cost` to observed `X_plot`. This is a fast development visualization step only. It is not used for training or quantitative evaluation, and final polished figures can later recompute PHATE jointly on observed and generated samples.

In [10]:
from sklearn.neighbors import KNeighborsRegressor

knn_neighbors = min(20, len(eb["X_cost"]))
phate_projector = KNeighborsRegressor(n_neighbors=knn_neighbors, weights="distance")
phate_projector.fit(np.asarray(eb["X_cost"], dtype=np.float32), np.asarray(eb["X_plot"], dtype=np.float32))

sample_n = min(1200 if QUICK_MODE else 2200, len(X0_val))
if SMOKE_MODE:
    sample_n = min(150, len(X0_val))
sample_idx = subsample_idx(len(X0_val), sample_n, seed=SEED + 31)
x0_eval_20d = torch.as_tensor(X0_val[sample_idx], dtype=torch.float32, device=DEVICE)

eb_model.eval()
euler_population_steps = 100 if not SMOKE_MODE else 25
with torch.no_grad():
    eb_final_20d_t, eb_traj_20d_t, eb_nfe = euler_sample(eb_model, x0_eval_20d, n_steps=euler_population_steps, return_traj=True)
eb_traj_20d = as_np(eb_traj_20d_t)
eb_final_20d = as_np(eb_final_20d_t)

show_times = [0.0, 0.25, 0.5, 0.75, 1.0]
show_steps = np.round(np.asarray(show_times) * (eb_traj_20d.shape[0] - 1)).astype(int)
generated_phate_by_time = [phate_projector.predict(eb_traj_20d[step]) for step in show_steps]

xlim, ylim = robust_limits(X_source_phate, X_target_phate, *generated_phate_by_time, margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(12.8, 2.85), sharex=True, sharey=True)
for ax, tau, gen_plot in zip(axes, show_times, generated_phate_by_time):
    if tau == 1.0:
        tidx = subsample_idx(len(X_target_phate), 800, seed=101)
        ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.65", alpha=0.20, linewidths=0, label="real target")
    ax.scatter(gen_plot[:, 0], gen_plot[:, 1], s=6, color="#2F6B5E", alpha=0.50, linewidths=0, label="generated")
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=f"t={tau:.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
add_note(axes[0], "20D Euler states\nPHATE display only", loc="lower left")
save_figure(fig, "fig03_08_eb_population_evolution_phate.png")
print({"sampling_space": "20D PC", "projection_for_plot_only": "kNN X_cost->X_plot", "euler_nfe": eb_nfe, "trajectory_shape": eb_traj_20d.shape})

{'sampling_space': '20D PC', 'projection_for_plot_only': 'kNN X_cost->X_plot', 'euler_nfe': 100, 'trajectory_shape': (101, 360, 20)}


## 7. Euler Step Sensitivity

This diagnostic fixes the trained 20D EB model and varies the Euler step count. Every generated endpoint is produced in 20D PC space.

The PHATE panels are only display projections. The CSV metrics are computed in 20D: MMD to the fine-step reference, endpoint MMD to held-out target cells, and Sliced W2 to held-out target cells.

In [11]:
euler_Ks = [1, 2, 5, 10, 20, 50, 100, 200]
reference_K = 200 if QUICK_MODE else 500
if SMOKE_MODE:
    euler_Ks = [1, 2, 5, 10, 20]
    reference_K = 20

with torch.no_grad():
    ref_endpoint_t, _, ref_nfe = euler_sample(eb_model, x0_eval_20d, n_steps=reference_K, return_traj=False)
ref_endpoint_20d = as_np(ref_endpoint_t)

target_eval_20d = X1_val.astype(np.float32)
euler_rows = []
euler_panels = []
for K in euler_Ks:
    tic = time.perf_counter()
    with torch.no_grad():
        endpoint_t, _, nfe = euler_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=False)
    wall = time.perf_counter() - tic
    endpoint_20d = as_np(endpoint_t)
    projected = phate_projector.predict(endpoint_20d)
    euler_panels.append({"K": K, "nfe": nfe, "endpoint_20d": endpoint_20d, "endpoint_phate": projected})
    euler_rows.append({
        "solver": "euler",
        "K": int(K),
        "nfe": int(nfe),
        "reference_K": int(reference_K),
        "wall_time_sec": float(wall),
        "mmd_20d_to_reference": float(mmd_rbf(endpoint_20d, ref_endpoint_20d)),
        "endpoint_mmd_20d_to_target_val": float(mmd_rbf(endpoint_20d, target_eval_20d)),
        "sliced_w2_20d_to_target_val": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + K)),
    })

euler_table = pd.DataFrame(euler_rows)
save_csv(euler_table, "ch03_euler_step_sensitivity.csv")

ncols = 4
nrows = int(np.ceil(len(euler_panels) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 3.0 * nrows), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(-1)
xlim, ylim = robust_limits(X_target_phate, *(p["endpoint_phate"] for p in euler_panels), margin=0.10)
for ax, panel in zip(axes, euler_panels):
    tidx = subsample_idx(len(X_target_phate), 650, seed=111)
    ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.68", alpha=0.18, linewidths=0)
    pts = panel["endpoint_phate"]
    ax.scatter(pts[:, 0], pts[:, 1], s=6, color="#2F6B5E", alpha=0.50, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=f"Euler K={panel['K']}\nNFE={panel['nfe']}")
for ax in axes[len(euler_panels):]:
    ax.axis("off")
fig.suptitle("Euler step sensitivity: 20D endpoints projected to PHATE for display")
save_figure(fig, "fig03_09_euler_step_sensitivity_phate.png")
euler_table

,solver,K,nfe,reference_K,wall_time_sec,mmd_20d_to_reference,endpoint_mmd_20d_to_target_val,sliced_w2_20d_to_target_val
0,euler,1,1,500,0.000330,0.075017,0.068530,0.855199
1,euler,2,2,500,0.001000,0.020687,0.018689,0.528526
2,euler,5,5,500,0.002515,0.003219,0.006708,0.478410
3,euler,10,10,500,0.002822,0.001095,0.006426,0.626000
4,euler,20,20,500,0.004842,0.000363,0.006606,0.695970
5,euler,50,50,500,0.011024,0.000066,0.006784,0.753696
6,euler,100,100,500,0.019752,0.000014,0.006854,0.814577
7,euler,200,200,500,0.041864,0.000002,0.006891,0.710712


## 8. Solver Comparison Lite

This first version compares fixed-step Euler, fixed-step midpoint, and `dopri5` when `torchdiffeq` is available. The comparison reports NFE, wall-clock time, MMD in 20D, and Sliced W2 in 20D.

This is not a CNF training baseline. It is only a post-training sampling diagnostic for the same trained 20D CFM vector field.

In [12]:
solver_rows = []
solver_specs = [("euler", K) for K in [5, 10, 20, 50, 100]] + [("midpoint", K) for K in [5, 10, 20, 50]]
if SMOKE_MODE:
    solver_specs = [("euler", K) for K in [5, 10]] + [("midpoint", K) for K in [5, 10]]

for solver_name, K in solver_specs:
    tic = time.perf_counter()
    if solver_name == "euler":
        endpoint_t, traj_t, nfe = euler_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=True)
    elif solver_name == "midpoint":
        endpoint_t, traj_t, nfe = midpoint_sample(eb_model, x0_eval_20d, n_steps=K, return_traj=True)
    else:
        raise ValueError(solver_name)
    wall = time.perf_counter() - tic
    endpoint_20d = as_np(endpoint_t)
    traj_20d = as_np(traj_t)
    solver_rows.append({
        "sampler": solver_name,
        "steps": int(K),
        "rtol": np.nan,
        "nfe": int(nfe),
        "wall_time_sec": float(wall),
        "mmd_20d": float(mmd_rbf(endpoint_20d, target_eval_20d)),
        "sliced_w2_20d": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + K + (0 if solver_name == "euler" else 1000))),
        "trajectory_straightness_20d": float(trajectory_straightness(traj_20d)),
    })

rtols = [1e-3, 1e-5]
if SMOKE_MODE:
    rtols = [1e-3]
for rtol in rtols:
    try:
        tic = time.perf_counter()
        endpoint_t, traj_t, nfe = odeint_sample(eb_model, x0_eval_20d, rtol=rtol, atol=rtol, method="dopri5")
        wall = time.perf_counter() - tic
        endpoint_20d = as_np(endpoint_t)
        traj_20d = as_np(traj_t)
        solver_rows.append({
            "sampler": "dopri5",
            "steps": "adaptive",
            "rtol": float(rtol),
            "nfe": int(nfe),
            "wall_time_sec": float(wall),
            "mmd_20d": float(mmd_rbf(endpoint_20d, target_eval_20d)),
            "sliced_w2_20d": float(sliced_wasserstein_distance(endpoint_20d, target_eval_20d, n_projections=128 if not SMOKE_MODE else 48, seed=SEED + int(-np.log10(rtol)))),
            "trajectory_straightness_20d": float(trajectory_straightness(traj_20d)),
        })
    except ImportError as exc:
        message = f"torchdiffeq unavailable; skipped dopri5 rtol={rtol}: {exc}"
        warnings.warn(message)
        skipped_items.append(message)
    except Exception as exc:
        message = f"dopri5 failed for rtol={rtol}: {type(exc).__name__}: {exc}"
        warnings.warn(message)
        skipped_items.append(message)

solver_table = pd.DataFrame(solver_rows)
save_csv(solver_table, "table03_01_solver_diagnostics.csv")
fig, ax = plot_nfe_vs_error(solver_table, y="mmd_20d")
ax.set_ylabel("MMD to target in 20D PC space")
ax.set_title("NFE versus endpoint error")
save_figure(fig, "fig03_10_nfe_vs_endpoint_error.png")
solver_table

,sampler,steps,rtol,nfe,wall_time_sec,mmd_20d,sliced_w2_20d,trajectory_straightness_20d
0,euler,5,NaN,5,0.002447,0.006708,0.478410,0.067777
1,euler,10,NaN,10,0.002888,0.006426,0.626000,0.061189
2,euler,20,NaN,20,0.004614,0.006606,0.695970,0.057071
3,euler,50,NaN,50,0.010477,0.006784,0.753696,0.054413
4,euler,100,NaN,100,0.020016,0.006854,0.814577,0.053472
5,midpoint,5,NaN,10,0.002768,0.006767,0.703954,0.053877
6,midpoint,10,NaN,20,0.004385,0.006874,0.784456,0.052912
7,midpoint,20,NaN,40,0.007999,0.006913,0.776727,0.052627
8,midpoint,50,NaN,100,0.019086,0.006926,0.857414,0.052535
9,dopri5,adaptive,0.00100,50,0.152387,0.006932,0.961883,0.000000


## 9. CNF-Endpoint Baseline: Same Endpoint Objective, ODE-in-the-Loop Training

This baseline uses the same EB 20D train/validation split and the same endpoint-pair source as CFM, but changes the training control flow.

- CFM trains by local velocity regression at sampled interpolation times and does not solve the learned ODE inside the loss.
- CNF-Endpoint predicts `x1_pred = Phi_theta^{0->1}(x0)` and minimizes endpoint MSE against the sampled target endpoint.
- CNF-Endpoint uses `torchdiffeq.odeint_adjoint` for backpropagation through the ODE solve.
- This is not FFJORD or likelihood training. CNF maximum-likelihood with divergence/Hutchinson trace estimation is a different objective and is intentionally left out of the main comparison.

All model states, endpoint losses, MMD, and Sliced W2 values in this section are computed in 20D PC space. PHATE appears only in the sample display figure.

In [13]:
cnf_endpoint_completed = False
cnf_endpoint_skipped_reason = None

def record_skip(item, reason):
    skipped_items.append({"item": str(item), "reason": str(reason)})


def make_seeded_pair_batch_fn(X0, X1, seed=42):
    local_rng = np.random.default_rng(seed)

    def _pair_batch_fn(batch_size):
        idx0 = local_rng.integers(0, len(X0), size=int(batch_size))
        idx1 = local_rng.integers(0, len(X1), size=int(batch_size))
        return {"x0": X0[idx0].astype(np.float32), "x1": X1[idx1].astype(np.float32)}

    return _pair_batch_fn


def endpoint_mmd_sliced_20d(samples_20d, target_20d, seed=42, n_projections=None):
    if n_projections is None:
        n_projections = 48 if SMOKE_MODE else 128
    return {
        "endpoint_mmd_20d": float(mmd_rbf(samples_20d, target_20d)),
        "sliced_w2_20d": float(sliced_wasserstein_distance(samples_20d, target_20d, n_projections=n_projections, seed=seed)),
    }


def eval_cfm_endpoint_20d(model, x0_np, target_np, n_steps=30, seed=42):
    model.eval()
    with torch.no_grad():
        x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=DEVICE)
        endpoint_t, _, nfe = euler_sample(model, x0_t, n_steps=int(n_steps), return_traj=False)
    endpoint_np = as_np(endpoint_t)
    metrics = endpoint_mmd_sliced_20d(endpoint_np, target_np, seed=seed)
    return endpoint_np, int(nfe), metrics


def eval_ode_endpoint_20d(model, x0_np, target_np, rtol=1e-3, seed=42):
    model.eval()
    with torch.no_grad():
        x0_t = torch.as_tensor(x0_np, dtype=torch.float32, device=DEVICE)
        endpoint_t, _, nfe = odeint_sample(model, x0_t, rtol=float(rtol), atol=float(rtol), method="dopri5")
    endpoint_np = as_np(endpoint_t)
    metrics = endpoint_mmd_sliced_20d(endpoint_np, target_np, seed=seed)
    return endpoint_np, int(nfe), metrics


class EndpointODEFunc(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.nfe = 0

    def reset_nfe(self):
        self.nfe = 0

    def forward(self, t, x):
        self.nfe += 1
        tt = torch.ones((x.shape[0], 1), device=x.device, dtype=x.dtype) * t
        return self.model(x, tt)


cnf_steps = 80 if QUICK_MODE else 600
cnf_batch_size = 64 if QUICK_MODE else 128
cnf_rtol = 1e-3 if QUICK_MODE else 1e-4
cnf_log_every = 20 if QUICK_MODE else 50
cfm_e1_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    cnf_steps = 6
    cnf_batch_size = 32
    cnf_rtol = 2e-3
    cnf_log_every = 3
    cfm_e1_sample_steps = 8

cnf_eval_n = min(360 if QUICK_MODE else 900, len(X0_val), len(X1_val))
if SMOKE_MODE:
    cnf_eval_n = min(90, len(X0_val), len(X1_val))
eval_idx0 = subsample_idx(len(X0_val), cnf_eval_n, seed=SEED + 200)
eval_target_idx = subsample_idx(len(X1_val), cnf_eval_n, seed=SEED + 201)
e1_eval_x0_20d = X0_val[eval_idx0].astype(np.float32)
e1_eval_target_20d = X1_val[eval_target_idx].astype(np.float32)

# Fair lightweight CFM probe for the training-cost curve. It uses the same architecture and pair source as CNF-Endpoint.
set_seed(SEED + 210)
cfm_e1_model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
cfm_e1_optimizer = torch.optim.Adam(cfm_e1_model.parameters(), lr=1e-3)
cfm_e1_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 211)
cfm_e1_rows = []
start = time.perf_counter()
last_time = start
last_step = 0
for step in range(1, cnf_steps + 1):
    batch = cfm_e1_pair_batch(cnf_batch_size)
    x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
    x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
    loss = cfm_loss_from_pairs(cfm_e1_model, x0, x1)
    cfm_e1_optimizer.zero_grad()
    loss.backward()
    cfm_e1_optimizer.step()
    if step == 1 or step % cnf_log_every == 0 or step == cnf_steps:
        now = time.perf_counter()
        _, sample_nfe, met = eval_cfm_endpoint_20d(cfm_e1_model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=cfm_e1_sample_steps, seed=SEED + step)
        cfm_e1_rows.append({
            "method": "CFM",
            "step": int(step),
            "wall_time_sec": float(now - start),
            "sec_per_step": float((now - last_time) / max(1, step - last_step)),
            "final_train_loss_20d": float(loss.detach().cpu()),
            "val_endpoint_mmd_20d": met["endpoint_mmd_20d"],
            "val_sliced_w2_20d": met["sliced_w2_20d"],
            "train_nfe_per_step": 1,
            "sample_nfe": int(sample_nfe),
        })
        last_time = now
        last_step = step

cnf_rows = []
cnf_model = None
try:
    from torchdiffeq import odeint_adjoint

    set_seed(SEED + 220)
    cnf_model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
    cnf_optimizer = torch.optim.Adam(cnf_model.parameters(), lr=1e-3)
    cnf_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 211)
    ode_func = EndpointODEFunc(cnf_model)
    t_grid_train = torch.tensor([0.0, 1.0], dtype=torch.float32, device=DEVICE)
    start = time.perf_counter()
    last_time = start
    last_step = 0
    for step in range(1, cnf_steps + 1):
        batch = cnf_pair_batch(cnf_batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        ode_func.reset_nfe()
        pred_traj = odeint_adjoint(
            ode_func,
            x0,
            t_grid_train,
            rtol=float(cnf_rtol),
            atol=float(cnf_rtol),
            method="dopri5",
            adjoint_params=tuple(cnf_model.parameters()),
        )
        x1_pred = pred_traj[-1]
        loss = ((x1_pred - x1) ** 2).mean(dim=-1).mean()
        cnf_optimizer.zero_grad()
        loss.backward()
        cnf_optimizer.step()
        train_nfe = int(ode_func.nfe)
        if step == 1 or step % cnf_log_every == 0 or step == cnf_steps:
            now = time.perf_counter()
            _, sample_nfe, met = eval_ode_endpoint_20d(cnf_model, e1_eval_x0_20d, e1_eval_target_20d, rtol=cnf_rtol, seed=SEED + 500 + step)
            cnf_rows.append({
                "method": "CNF-Endpoint",
                "step": int(step),
                "wall_time_sec": float(now - start),
                "sec_per_step": float((now - last_time) / max(1, step - last_step)),
                "final_train_loss_20d": float(loss.detach().cpu()),
                "val_endpoint_mmd_20d": met["endpoint_mmd_20d"],
                "val_sliced_w2_20d": met["sliced_w2_20d"],
                "train_nfe_per_step": int(train_nfe),
                "sample_nfe": int(sample_nfe),
            })
            last_time = now
            last_step = step
    cnf_endpoint_completed = True
except Exception as exc:
    cnf_endpoint_skipped_reason = f"{type(exc).__name__}: {exc}"
    record_skip("CNF-Endpoint baseline", cnf_endpoint_skipped_reason)

cfm_e1_history = pd.DataFrame(cfm_e1_rows)
cnf_e1_history = pd.DataFrame(cnf_rows)
e1_history = pd.concat([cfm_e1_history, cnf_e1_history], ignore_index=True)

# Final sample metrics and display endpoints for the E1 table/figure.
cfm_e1_endpoint_20d, cfm_e1_sample_nfe, cfm_e1_metrics = eval_cfm_endpoint_20d(
    cfm_e1_model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=cfm_e1_sample_steps, seed=SEED + 230
)
summary_rows = [{
    "method": "CFM",
    "train_objective": "local_velocity_regression_20D",
    "train_uses_ode": False,
    "steps": int(cnf_steps),
    "batch_size": int(cnf_batch_size),
    "wall_time_sec": float(cfm_e1_history["wall_time_sec"].iloc[-1]),
    "mean_sec_per_step": float(cfm_e1_history["wall_time_sec"].iloc[-1] / max(cnf_steps, 1)),
    "train_nfe_per_step": 1,
    "final_train_loss_20d": float(cfm_e1_history["final_train_loss_20d"].iloc[-1]),
    "endpoint_mmd_20d": cfm_e1_metrics["endpoint_mmd_20d"],
    "sliced_w2_20d": cfm_e1_metrics["sliced_w2_20d"],
    "sample_nfe": int(cfm_e1_sample_nfe),
}]

if cnf_endpoint_completed and cnf_model is not None:
    cnf_e1_endpoint_20d, cnf_e1_sample_nfe, cnf_e1_metrics = eval_ode_endpoint_20d(
        cnf_model, e1_eval_x0_20d, e1_eval_target_20d, rtol=cnf_rtol, seed=SEED + 231
    )
    summary_rows.append({
        "method": "CNF-Endpoint",
        "train_objective": "endpoint_mse_after_adjoint_ode_solve_20D",
        "train_uses_ode": True,
        "steps": int(cnf_steps),
        "batch_size": int(cnf_batch_size),
        "wall_time_sec": float(cnf_e1_history["wall_time_sec"].iloc[-1]),
        "mean_sec_per_step": float(cnf_e1_history["wall_time_sec"].iloc[-1] / max(cnf_steps, 1)),
        "train_nfe_per_step": float(cnf_e1_history["train_nfe_per_step"].mean()),
        "final_train_loss_20d": float(cnf_e1_history["final_train_loss_20d"].iloc[-1]),
        "endpoint_mmd_20d": cnf_e1_metrics["endpoint_mmd_20d"],
        "sliced_w2_20d": cnf_e1_metrics["sliced_w2_20d"],
        "sample_nfe": int(cnf_e1_sample_nfe),
    })
else:
    cnf_e1_endpoint_20d = None
    cnf_e1_sample_nfe = None

E1_table = pd.DataFrame(summary_rows)
save_csv(E1_table, "tableE1_cfm_vs_cnf_endpoint.csv")

fig, ax = plt.subplots(figsize=(5.6, 3.5))
for method, group in e1_history.groupby("method", sort=False):
    ax.plot(group["wall_time_sec"], group["val_endpoint_mmd_20d"], marker="o", linewidth=1.5, markersize=3.0, label=method)
ax.set_xlabel("training wall-clock seconds")
ax.set_ylabel("validation endpoint MMD in 20D")
ax.set_title("CFM local regression versus CNF-Endpoint adjoint training")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False)
save_figure(fig, "figE1_cfm_vs_cnf_endpoint_training_cost.png")

if cnf_e1_endpoint_20d is not None:
    cfm_plot = phate_projector.predict(cfm_e1_endpoint_20d)
    cnf_plot = phate_projector.predict(cnf_e1_endpoint_20d)
    target_plot = phate_projector.predict(e1_eval_target_20d)
    xlim, ylim = robust_limits(target_plot, cfm_plot, cnf_plot, margin=0.10)
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.8), sharex=True, sharey=True)
    panels = [(axes[0], cfm_plot, "A. CFM endpoints\nPHATE display only", "#2F6B5E"), (axes[1], cnf_plot, "B. CNF-Endpoint endpoints\nPHATE display only", "#B9352B")]
    for ax, pts, title, color in panels:
        ax.scatter(target_plot[:, 0], target_plot[:, 1], s=7, color="0.65", alpha=0.24, linewidths=0, label="held-out target")
        ax.scatter(pts[:, 0], pts[:, 1], s=8, color=color, alpha=0.56, linewidths=0, label="generated")
        format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=title)
        ax.legend(frameon=False, loc="best")
        if ax is not axes[0]:
            ax.set_ylabel("")
    fig.suptitle("Endpoint samples from 20D models projected for PHATE visualization")
    save_figure(fig, "figE1_cfm_vs_cnf_endpoint_samples_phate.png")
else:
    record_skip("figE1_cfm_vs_cnf_endpoint_samples_phate.png", "CNF-Endpoint did not complete")

E1_table

,method,train_objective,train_uses_ode,steps,batch_size,wall_time_sec,mean_sec_per_step,train_nfe_per_step,final_train_loss_20d,endpoint_mmd_20d,sliced_w2_20d,sample_nfe
0,CFM,local_velocity_regression_20D,False,600,128,1.499027,0.002498,1.000000,5.157436,0.009816,0.431492,50
1,CNF-Endpoint,endpoint_mse_after_adjoint_ode_solve_20D,True,600,128,53.463115,0.089105,101.384615,5.346793,0.877996,2.105614,68


## 10. Time Sampling Strategy Ablation

This ablation changes only the training-time interpolation-time distribution. All models are trained on the same EB 20D train/validation split with the same architecture and endpoint-pair source.

The empirical distribution plot is sampled directly from the time samplers used by the training loop. Metrics and validation MSE are in 20D PC space.

In [14]:
time_sampling_completed = False

time_strategy_specs = [
    ("uniform", {}),
    ("logit_normal_sigma_0.5", {"sigma": 0.5}),
    ("logit_normal_sigma_1.0", {"sigma": 1.0}),
    ("logit_normal_sigma_2.0", {"sigma": 2.0}),
    ("beta_2_2", {"alpha": 2.0, "beta": 2.0}),
    ("beta_0.5_0.5", {"alpha": 0.5, "beta": 0.5}),
    ("cosine", {}),
]


def sample_t_numpy(strategy, n, seed=42):
    local_rng = np.random.default_rng(seed)
    if strategy == "uniform":
        t = local_rng.random(n)
    elif strategy.startswith("logit_normal"):
        sigma = dict(time_strategy_specs)[strategy]["sigma"]
        z = local_rng.normal(loc=0.0, scale=float(sigma), size=n)
        t = 1.0 / (1.0 + np.exp(-z))
    elif strategy == "beta_2_2":
        t = local_rng.beta(2.0, 2.0, size=n)
    elif strategy == "beta_0.5_0.5":
        t = local_rng.beta(0.5, 0.5, size=n)
    elif strategy == "cosine":
        u = local_rng.random(n)
        t = 0.5 * (1.0 - np.cos(np.pi * u))
    else:
        raise ValueError(strategy)
    return np.clip(t.astype(np.float32), 1e-4, 1.0 - 1e-4)


def sample_t_torch(strategy, batch_size, device, seed=None):
    # Numpy-backed sampling keeps every strategy exactly aligned with the empirical-density plot.
    if seed is None:
        seed = np.random.randint(0, 2**31 - 1)
    arr = sample_t_numpy(strategy, int(batch_size), seed=seed)
    return torch.as_tensor(arr[:, None], dtype=torch.float32, device=device)


def train_cfm_with_time_strategy(strategy, steps, batch_size, log_every, seed_offset=0):
    set_seed(SEED + 300 + seed_offset)
    model = VelocityMLP(x_dim=20, hidden_dim=256, hidden_layers=4).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    local_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 310 + seed_offset)
    rows = []
    start = time.perf_counter()
    last_time = start
    last_step = 0
    for step in range(1, int(steps) + 1):
        batch = local_pair_batch(batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        t_batch = sample_t_torch(strategy, batch_size, DEVICE, seed=SEED + 320 + seed_offset * 100000 + step)
        loss = cfm_loss_from_pairs(model, x0, x1, s=t_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step == 1 or step % int(log_every) == 0 or step == int(steps):
            now = time.perf_counter()
            val_mse = val_cfm_mse(model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=900 if not SMOKE_MODE else 120)
            rows.append({
                "strategy": strategy,
                "step": int(step),
                "train_loss_20d": float(loss.detach().cpu()),
                "val_mse_20d": float(val_mse),
                "wall_time_sec": float(now - start),
                "sec_per_step": float((now - last_time) / max(1, step - last_step)),
            })
            last_time = now
            last_step = step
    return model, pd.DataFrame(rows)

# Empirical t distributions.
fig, ax = plt.subplots(figsize=(6.5, 3.7))
bins = np.linspace(0.0, 1.0, 55)
for i, (strategy, _) in enumerate(time_strategy_specs):
    samples = sample_t_numpy(strategy, 8000 if not SMOKE_MODE else 1200, seed=SEED + 330 + i)
    hist, edges = np.histogram(samples, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, hist, linewidth=1.4, label=strategy)
ax.set_xlabel("sampled interpolation time t")
ax.set_ylabel("empirical density")
ax.set_title("Training-time sampling strategies for CFM")
ax.legend(frameon=False, fontsize=6, ncol=2)
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE3_time_sampling_distributions.png")

time_steps = 350 if QUICK_MODE else 1200
time_batch_size = 192 if QUICK_MODE else 256
time_log_every = 70 if QUICK_MODE else 150
time_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    time_steps = 8
    time_batch_size = 64
    time_log_every = 4
    time_sample_steps = 8

time_histories = []
time_summary_rows = []
for si, (strategy, _) in enumerate(time_strategy_specs):
    model, hist = train_cfm_with_time_strategy(strategy, time_steps, time_batch_size, time_log_every, seed_offset=si)
    endpoint_20d, sample_nfe, met = eval_cfm_endpoint_20d(model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=time_sample_steps, seed=SEED + 400 + si)
    hist["endpoint_mmd_20d"] = np.nan
    hist.loc[hist.index[-1], "endpoint_mmd_20d"] = met["endpoint_mmd_20d"]
    time_histories.append(hist)
    time_summary_rows.append({
        "strategy": strategy,
        "steps": int(time_steps),
        "final_train_loss_20d": float(hist["train_loss_20d"].iloc[-1]),
        "final_val_mse_20d": float(hist["val_mse_20d"].iloc[-1]),
        "endpoint_mmd_20d": met["endpoint_mmd_20d"],
        "sliced_w2_20d": met["sliced_w2_20d"],
        "sample_nfe": int(sample_nfe),
        "wall_time_sec": float(hist["wall_time_sec"].iloc[-1]),
    })

time_history = pd.concat(time_histories, ignore_index=True)
time_table = pd.DataFrame(time_summary_rows)
save_csv(time_table, "tableE3_time_sampling_ablation.csv")

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for strategy, group in time_history.groupby("strategy", sort=False):
    ax.plot(group["step"], group["val_mse_20d"], linewidth=1.35, label=strategy)
ax.set_xlabel("training step")
ax.set_ylabel("validation CFM MSE in 20D")
ax.set_title("Time sampling ablation: validation velocity-regression MSE")
ax.grid(axis="y", color="0.92", linewidth=0.7)
ax.legend(frameon=False, fontsize=6, ncol=2)
save_figure(fig, "figE3_time_sampling_val_mse.png")

fig, ax = plt.subplots(figsize=(6.6, 3.6))
order = time_table.sort_values("endpoint_mmd_20d")["strategy"].tolist()
sns.barplot(data=time_table, x="strategy", y="endpoint_mmd_20d", order=order, ax=ax, color="#2F6B5E")
ax.set_xlabel("time sampling strategy")
ax.set_ylabel("endpoint MMD in 20D")
ax.set_title("Final endpoint MMD after short CFM runs")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE3_time_sampling_endpoint_mmd.png")

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6))
sns.barplot(data=time_table, x="strategy", y="endpoint_mmd_20d", order=order, ax=axes[0], color="#2F6B5E")
sns.barplot(data=time_table, x="strategy", y="sliced_w2_20d", order=order, ax=axes[1], color="#B9352B")
for ax, ylabel in zip(axes, ["endpoint MMD in 20D", "Sliced W2 in 20D"]):
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=40)
    ax.grid(axis="y", color="0.92", linewidth=0.7)
axes[0].set_title("MMD")
axes[1].set_title("Sliced W2")
fig.suptitle("Time sampling ablation final endpoint metrics")
save_figure(fig, "figE3_time_sampling_final_bar.png")
time_sampling_completed = True
time_table

,strategy,steps,final_train_loss_20d,final_val_mse_20d,endpoint_mmd_20d,sliced_w2_20d,sample_nfe,wall_time_sec
0,uniform,1200,4.268666,4.543735,0.007025,0.449011,50,2.059240
1,logit_normal_sigma_0.5,1200,4.110121,5.040607,0.007857,0.677951,50,2.071338
2,logit_normal_sigma_1.0,1200,4.478451,4.651358,0.006162,0.397897,50,2.069029
3,logit_normal_sigma_2.0,1200,5.155850,4.629635,0.008528,0.730765,50,2.076655
4,beta_2_2,1200,4.248035,4.715159,0.006425,0.485969,50,2.065672
5,beta_0.5_0.5,1200,4.953589,4.580538,0.006989,0.555961,50,2.069864
6,cosine,1200,4.811807,4.479515,0.007193,0.428569,50,2.065518


## 11. Network Capacity Ablation

This ablation changes only the `VelocityMLP` capacity. Input and output dimensions remain 20, and every model uses the same EB 20D train/validation split and random endpoint-pair source.

Metrics are final validation CFM MSE, endpoint MMD in 20D, Sliced W2 in 20D, parameter count, and wall-clock time.

In [15]:
capacity_ablation_completed = False

capacity_configs = [
    {"config": "tiny", "hidden_dim": 64, "hidden_layers": 3},
    {"config": "small", "hidden_dim": 128, "hidden_layers": 4},
    {"config": "medium", "hidden_dim": 256, "hidden_layers": 4},
]
if not QUICK_MODE:
    capacity_configs.append({"config": "large", "hidden_dim": 512, "hidden_layers": 6})

capacity_steps = 300 if QUICK_MODE else 1200
capacity_batch_size = 192 if QUICK_MODE else 256
capacity_log_every = 100 if QUICK_MODE else 200
capacity_sample_steps = 30 if QUICK_MODE else 50
if SMOKE_MODE:
    capacity_steps = 8
    capacity_batch_size = 64
    capacity_log_every = 4
    capacity_sample_steps = 8

capacity_rows = []
capacity_histories = []
for ci, cfg in enumerate(capacity_configs):
    set_seed(SEED + 500 + ci)
    model = VelocityMLP(x_dim=20, hidden_dim=cfg["hidden_dim"], hidden_layers=cfg["hidden_layers"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    local_pair_batch = make_seeded_pair_batch_fn(X0_train, X1_train, seed=SEED + 510 + ci)
    start = time.perf_counter()
    last_loss = np.nan
    hist_rows = []
    for step in range(1, int(capacity_steps) + 1):
        batch = local_pair_batch(capacity_batch_size)
        x0 = torch.as_tensor(batch["x0"], dtype=torch.float32, device=DEVICE)
        x1 = torch.as_tensor(batch["x1"], dtype=torch.float32, device=DEVICE)
        loss = cfm_loss_from_pairs(model, x0, x1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        if step == 1 or step % capacity_log_every == 0 or step == int(capacity_steps):
            val_mse = val_cfm_mse(model, val_x0, val_x1, val_t_grid, device=DEVICE, max_eval_pairs=900 if not SMOKE_MODE else 120)
            hist_rows.append({"config": cfg["config"], "step": int(step), "train_loss_20d": last_loss, "val_mse_20d": val_mse})
    wall = time.perf_counter() - start
    endpoint_20d, sample_nfe, met = eval_cfm_endpoint_20d(model, e1_eval_x0_20d, e1_eval_target_20d, n_steps=capacity_sample_steps, seed=SEED + 540 + ci)
    hist = pd.DataFrame(hist_rows)
    capacity_histories.append(hist)
    capacity_rows.append({
        "config": cfg["config"],
        "input_dim": 20,
        "output_dim": 20,
        "hidden_dim": int(cfg["hidden_dim"]),
        "hidden_layers": int(cfg["hidden_layers"]),
        "n_parameters": int(count_parameters(model)),
        "lr": 1e-3,
        "batch_size": int(capacity_batch_size),
        "steps": int(capacity_steps),
        "final_train_mse_20d": float(last_loss),
        "final_val_mse_20d": float(hist["val_mse_20d"].iloc[-1]),
        "endpoint_mmd_20d": met["endpoint_mmd_20d"],
        "sliced_w2_20d": met["sliced_w2_20d"],
        "sample_nfe": int(sample_nfe),
        "wall_time_sec": float(wall),
    })

capacity_table = pd.DataFrame(capacity_rows)
capacity_history = pd.concat(capacity_histories, ignore_index=True)
save_csv(capacity_table, "tableT2_training_hyperparams_capacity.csv")

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(capacity_table["n_parameters"], capacity_table["final_val_mse_20d"], marker="o", linewidth=1.5, color="#2F6B5E")
for _, row in capacity_table.iterrows():
    ax.text(row["n_parameters"], row["final_val_mse_20d"], row["config"], fontsize=7, ha="left", va="bottom")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("final validation MSE in 20D")
ax.set_title("Capacity ablation: validation velocity-regression error")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE2_capacity_val_mse.png")

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(capacity_table["n_parameters"], capacity_table["endpoint_mmd_20d"], marker="s", linewidth=1.5, color="#B9352B")
for _, row in capacity_table.iterrows():
    ax.text(row["n_parameters"], row["endpoint_mmd_20d"], row["config"], fontsize=7, ha="left", va="bottom")
ax.set_xscale("log")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("endpoint MMD in 20D")
ax.set_title("Capacity ablation: endpoint distribution error")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE2_capacity_endpoint_mmd.png")
capacity_ablation_completed = True
capacity_table

,config,input_dim,output_dim,hidden_dim,hidden_layers,n_parameters,lr,batch_size,steps,final_train_mse_20d,final_val_mse_20d,endpoint_mmd_20d,sliced_w2_20d,sample_nfe,wall_time_sec
0,tiny,20,20,64,3,11028,0.001,256,1200,5.229878,5.011684,0.008179,0.375417,50,1.787434
1,small,20,20,128,4,54932,0.001,256,1200,4.875462,4.718894,0.006995,0.371575,50,2.034561
2,medium,20,20,256,4,208148,0.001,256,1200,4.829438,4.621738,0.006655,0.393808,50,1.984651
3,large,20,20,512,6,1334804,0.001,256,1200,4.611034,4.728959,0.007487,0.464953,50,2.641509


## 12. Trajectory Straightness in 20D

This diagnostic uses the main EB CFM model and measures trajectory straightness in 20D PC space. The ratio is

`sum_k ||x_{k+1}-x_k|| / ||x_K-x_0||`.

A value near 1 indicates a nearly straight numerical path in 20D. PHATE is used only to display representative trajectories after the 20D straightness scores are computed.

In [16]:
straightness_completed = False


def per_trajectory_straightness(traj, eps=1e-8):
    traj = np.asarray(traj, dtype=float)
    if traj.ndim != 3:
        raise ValueError("traj must have shape (T, N, D)")
    steps = np.linalg.norm(np.diff(traj, axis=0), axis=-1)
    path_len = steps.sum(axis=0)
    endpoint = np.linalg.norm(traj[-1] - traj[0], axis=-1)
    ratio = path_len / np.maximum(endpoint, float(eps))
    return path_len, endpoint, ratio

straight_sample_n = min(900 if QUICK_MODE else 1600, len(X0_val))
straight_steps = 50 if QUICK_MODE else 100
if SMOKE_MODE:
    straight_sample_n = min(120, len(X0_val))
    straight_steps = 10
straight_idx = subsample_idx(len(X0_val), straight_sample_n, seed=SEED + 600)
straight_x0_t = torch.as_tensor(X0_val[straight_idx], dtype=torch.float32, device=DEVICE)
eb_model.eval()
with torch.no_grad():
    straight_endpoint_t, straight_traj_t, straight_nfe = euler_sample(eb_model, straight_x0_t, n_steps=straight_steps, return_traj=True)
straight_traj_20d = as_np(straight_traj_t)
path_length_20d, endpoint_distance_20d, straight_ratio = per_trajectory_straightness(straight_traj_20d)
q25, q50, q75, q90 = np.quantile(straight_ratio, [0.25, 0.50, 0.75, 0.90])
quantile_group = np.full(straight_ratio.shape, "middle", dtype=object)
quantile_group[straight_ratio <= q25] = "lower_quantile"
quantile_group[straight_ratio >= q75] = "upper_quantile"
straight_table = pd.DataFrame({
    "cell_index": straight_idx.astype(int),
    "endpoint_distance_20d": endpoint_distance_20d,
    "path_length_20d": path_length_20d,
    "straightness_ratio_20d": straight_ratio,
    "quantile_group": quantile_group,
})
save_csv(straight_table, "tableE5_trajectory_straightness.csv")

fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.hist(straight_ratio, bins=36, color="#2F6B5E", alpha=0.78, edgecolor="white", linewidth=0.4)
for value, label, color in [(straight_ratio.mean(), "mean", "#2F6B5E"), (q50, "median", "0.20"), (q90, "90th pct", "#B9352B")]:
    ax.axvline(value, color=color, linestyle="--", linewidth=1.2)
    ax.text(value, ax.get_ylim()[1] * 0.92, label, rotation=90, va="top", ha="right", fontsize=7, color=color)
ax.set_xlabel("20D straightness ratio")
ax.set_ylabel("trajectory count")
ax.set_title("Trajectory straightness under main EB CFM sampler")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE5_straightness_hist.png")

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(endpoint_distance_20d, straight_ratio, s=9, color="#2F6B5E", alpha=0.38, linewidths=0)
ax.set_xlabel("endpoint distance in 20D")
ax.set_ylabel("straightness ratio")
ax.set_title("Endpoint displacement versus 20D path straightness")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "figE5_endpoint_distance_vs_straightness.png")

low_candidates = np.flatnonzero(straight_ratio <= q25)
high_candidates = np.flatnonzero(straight_ratio >= q75)
low_show = low_candidates[subsample_idx(len(low_candidates), min(8, len(low_candidates)), seed=SEED + 610)]
high_show = high_candidates[subsample_idx(len(high_candidates), min(8, len(high_candidates)), seed=SEED + 611)]
fig, ax = plt.subplots(figsize=(5.8, 4.6))
tidx = subsample_idx(len(X_target_phate), 700, seed=SEED + 612)
ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color="0.70", alpha=0.18, linewidths=0, label="real target")
traj_phate_arrays = []
for j in low_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="#2F6B5E", alpha=0.58, linewidth=1.0)
    ax.scatter(pts[[0, -1], 0], pts[[0, -1], 1], s=14, color="#2F6B5E", alpha=0.70, linewidths=0)
for j in high_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="#B9352B", alpha=0.60, linewidth=1.0)
    ax.scatter(pts[[0, -1], 0], pts[[0, -1], 1], s=14, color="#B9352B", alpha=0.72, linewidths=0)
all_traj_plot = np.vstack(traj_phate_arrays) if traj_phate_arrays else X_target_phate[tidx]
xlim, ylim = robust_limits(X_target_phate[tidx], all_traj_plot, margin=0.10)
format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title="Representative 20D trajectories projected to PHATE display only")
ax.plot([], [], color="#2F6B5E", label="lower straightness quantile")
ax.plot([], [], color="#B9352B", label="upper straightness quantile")
ax.legend(frameon=False, loc="best")
save_figure(fig, "figE5_representative_trajectories_phate.png")
straightness_completed = True
straight_table.describe(include="all")

,cell_index,endpoint_distance_20d,path_length_20d,straightness_ratio_20d,quantile_group
count,360.000000,360.000000,360.000000,360.000000,360
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,middle
freq,NaN,NaN,NaN,NaN,180
mean,179.500000,13.680934,14.269862,1.053472,NaN
std,104.067286,4.036548,3.856292,0.058292,NaN
min,0.000000,4.635664,5.502244,1.006160,NaN
25%,89.750000,10.622012,11.447158,1.022562,NaN
50%,179.500000,13.665377,14.129058,1.034105,NaN
75%,269.250000,16.281289,16.708357,1.062406,NaN


## 13. Paper-Ready Figure and Table Export Pass

This section does not rerun or alter the experiments. It reuses the already computed 20D results and PHATE display projections to export cleaner paper-ready versions of the same figures, write PDF copies, and create rounded LaTeX-friendly tables.

The raw CSV files remain unchanged for provenance. The `paper_table*.csv/.tex/.md` files are formatted for paper drafting.

In [17]:
paper_ready_pass_completed = False
figure_quality_notes = {}

set_paper_style()

# --- Core toy/supporting figures ---
fig, ax = plt.subplots(figsize=(4.5, 2.9))
ax.plot(toy_history["step"], toy_history["loss"], color=PAPER_COLORS["generated"], linewidth=1.45)
ax.set_xlabel("Optimization step")
ax.set_ylabel("CFM MSE")
ax.set_title("Toy CFM loss")
clean_spines(ax, grid_axis="y")
save_paper_figure(fig, "fig_toy_loss")

xlim, ylim = robust_limits(toy_X0, toy_X1, toy_traj.reshape(-1, 2), margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(11.2, 2.45), sharex=True, sharey=True)
for pi, (ax, tau, step) in enumerate(zip(axes, toy_times, toy_step_idx)):
    pts = toy_traj[step]
    tidx = subsample_idx(len(toy_X1), 350, seed=61)
    ax.scatter(toy_X1[tidx, 0], toy_X1[tidx, 1], s=4.5, color=PAPER_COLORS["target"], alpha=0.18, linewidths=0)
    ax.scatter(pts[:, 0], pts[:, 1], s=5.5, color=PAPER_COLORS["generated"], alpha=0.58, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2", title=f"t={tau:.2f}")
    if pi != 0:
        ax.set_ylabel("")
    else:
        add_panel_label(ax, "A", x=-0.12)
    if pi < len(axes) - 1:
        ax.set_xlabel("")
save_paper_figure(fig, "fig_toy_evolution")

fig, toy_fig32_info = plot_toy_conditional_vs_marginal(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 4)
ax = fig.axes[0]
ax.set_title("Conditional velocities average into a marginal field")
ax.legend(frameon=False, loc="upper right", fontsize=7)
save_paper_figure(fig, "fig03_02_conditional_vs_marginal_toy")

fig = plot_toy_cfm_object_hierarchy(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 5)
axes = fig.axes[:3]
for label, title, ax in zip(["A", "B", "C"], ["One endpoint pair", "Many conditional paths", "Learned marginal field"], axes):
    ax.set_title(title)
    add_panel_label(ax, label)
axes[1].set_ylabel("")
axes[2].set_ylabel("")
fig.suptitle("")
save_paper_figure(fig, "fig03_03_cfm_object_hierarchy_toy")

# --- Main EB figures ---
fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(eb_history["step"], eb_history["train_loss_20d"], marker="o", markersize=2.2, linewidth=1.2, alpha=0.45, color=PAPER_COLORS["cfm"], label="train")
ax.plot(eb_history["step"], eb_history["val_mse_20d_fixed_t_grid"], marker="s", markersize=2.2, linewidth=1.2, alpha=0.55, color=PAPER_COLORS["cnf"], label="val")
if len(eb_history) >= 5:
    ax.plot(eb_history["step"], eb_history["train_loss_20d"].rolling(3, min_periods=1).mean(), color=PAPER_COLORS["cfm"], linewidth=1.55)
    ax.plot(eb_history["step"], eb_history["val_mse_20d_fixed_t_grid"].rolling(3, min_periods=1).mean(), color=PAPER_COLORS["cnf"], linewidth=1.55)
ax.set_xlabel("Optimization step")
ax.set_ylabel("Velocity-regression MSE (20D PCs)")
ax.set_title("CFM training and validation loss")
clean_spines(ax, grid_axis="y")
ax.legend(frameon=False, loc="best")
save_paper_figure(fig, "figB1_eb20d_train_val_loss")

fig = plot_endpoint_pairs_phate(X0_train_phate, X1_train_phate, n_pairs=pair_plot_n, seed=SEED + 30)
ax = fig.axes[0]
ax.set_title("Endpoint pairs shown in PHATE")
for line in ax.lines:
    line.set_alpha(0.18)
    line.set_linewidth(0.45)
ax.legend(frameon=False, loc="upper right", fontsize=7)
add_note(ax, "Training/pairing in 20D PCs\nPHATE display only", loc="lower left")
save_paper_figure(fig, "fig03_04_eb_endpoint_pairs_phate")

xlim, ylim = robust_limits(X_source_phate, X_target_phate, *generated_phate_by_time, margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(11.4, 2.65), sharex=True, sharey=True)
for pi, (ax, tau, gen_plot) in enumerate(zip(axes, show_times, generated_phate_by_time)):
    if tau == 1.0:
        tidx = subsample_idx(len(X_target_phate), 800, seed=101)
        ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=5, color=PAPER_COLORS["target"], alpha=0.30, linewidths=0, label="target")
    ax.scatter(gen_plot[:, 0], gen_plot[:, 1], s=5.5, color=PAPER_COLORS["generated"], alpha=0.58, linewidths=0, label="generated")
    format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=f"t={tau:.2f}")
    if pi != 0:
        ax.set_ylabel("")
    else:
        add_panel_label(ax, "A", x=-0.12)
    if pi < len(axes) - 1:
        ax.set_xlabel("")
add_note(axes[0], "20D Euler states\nPHATE display only", loc="lower left")
axes[-1].legend(frameon=False, loc="upper right", fontsize=7)
fig.subplots_adjust(top=0.88, wspace=0.08)
save_paper_figure(fig, "fig03_08_eb_population_evolution_phate")

ncols = 4
nrows = int(np.ceil(len(euler_panels) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(10.8, 5.2), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(-1)
xlim, ylim = robust_limits(X_target_phate, *(p["endpoint_phate"] for p in euler_panels), margin=0.10)
for idx, (ax, panel) in enumerate(zip(axes, euler_panels)):
    tidx = subsample_idx(len(X_target_phate), 650, seed=111)
    ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=4.5, color=PAPER_COLORS["target"], alpha=0.23, linewidths=0)
    pts = panel["endpoint_phate"]
    ax.scatter(pts[:, 0], pts[:, 1], s=5.2, color=PAPER_COLORS["generated"], alpha=0.58, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="PHATE 1" if idx // ncols == nrows - 1 else "", ylabel="PHATE 2" if idx % ncols == 0 else "", title=f"K={panel['K']}\nNFE={panel['nfe']}")
    add_panel_label(ax, chr(ord("A") + idx), x=-0.10, y=1.07)
for ax in axes[len(euler_panels):]:
    ax.axis("off")
add_note(axes[0], "Metrics: 20D PCs\nPHATE display only", loc="lower left")
fig.suptitle("Euler step sensitivity", y=1.02, fontsize=9.5)
save_paper_figure(fig, "fig03_09_euler_step_sensitivity_phate")

fig, ax = plt.subplots(figsize=(4.8, 3.25))
solver_colors = {"euler": PAPER_COLORS["euler"], "midpoint": PAPER_COLORS["midpoint"], "dopri5": PAPER_COLORS["dopri5"]}
solver_markers = {"euler": "o", "midpoint": "s", "dopri5": "^"}
for sampler, group in solver_table.groupby("sampler", sort=False):
    group = group.sort_values("nfe")
    ax.plot(group["nfe"], group["mmd_20d"], marker=solver_markers.get(sampler, "o"), color=solver_colors.get(sampler, "0.3"), linewidth=1.45, markersize=3.5, label=sampler)
ax.set_xlabel("Number of function evaluations (NFE)")
ax.set_ylabel("MMD to target (20D PCs)")
ax.set_title("NFE vs endpoint MMD")
clean_spines(ax, grid_axis="y")
ax.legend(frameon=False, loc="best")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.3f}"))
save_paper_figure(fig, "fig03_10_nfe_vs_endpoint_error")

# --- E1 diagnostics ---
fig, ax = plt.subplots(figsize=(5.0, 3.25))
for method, group in e1_history.groupby("method", sort=False):
    color = PAPER_COLORS["cfm"] if method == "CFM" else PAPER_COLORS["cnf"]
    ax.plot(group["wall_time_sec"], group["val_endpoint_mmd_20d"], marker="o", linewidth=1.45, markersize=3.2, color=color, label=method)
ax.set_xlabel("Wall-clock time (s)")
ax.set_ylabel("Endpoint MMD (20D PCs)")
ax.set_title("CFM vs CNF-Endpoint training cost")
clean_spines(ax, grid_axis="y")
ax.legend(frameon=False, loc="best")
if QUICK_MODE:
    add_note(ax, "QUICK_MODE diagnostic", loc="upper right")
save_paper_figure(fig, "figE1_cfm_vs_cnf_endpoint_training_cost")

if 'cnf_e1_endpoint_20d' in globals() and cnf_e1_endpoint_20d is not None:
    cfm_plot = phate_projector.predict(cfm_e1_endpoint_20d)
    cnf_plot = phate_projector.predict(cnf_e1_endpoint_20d)
    target_plot = phate_projector.predict(e1_eval_target_20d)
    xlim, ylim = robust_limits(target_plot, cfm_plot, cnf_plot, margin=0.10)
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.35), sharex=True, sharey=True)
    for label, ax, pts, title, color in [("A", axes[0], cfm_plot, "CFM endpoints", PAPER_COLORS["cfm"]), ("B", axes[1], cnf_plot, "CNF-Endpoint endpoints", PAPER_COLORS["cnf"] )]:
        ax.scatter(target_plot[:, 0], target_plot[:, 1], s=6, color=PAPER_COLORS["target"], alpha=0.28, linewidths=0, label="target")
        ax.scatter(pts[:, 0], pts[:, 1], s=7, color=color, alpha=0.62, linewidths=0, label="generated")
        format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title=title)
        add_panel_label(ax, label)
        ax.legend(frameon=False, loc="best", fontsize=7)
        if ax is axes[1]:
            ax.set_ylabel("")
    add_note(axes[0], "PHATE display only", loc="lower left")
    save_paper_figure(fig, "figE1_cfm_vs_cnf_endpoint_samples_phate")

# --- E2 capacity ---
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(capacity_table["n_parameters"], capacity_table["final_val_mse_20d"], marker="o", linewidth=1.45, color=PAPER_COLORS["generated"])
for _, row in capacity_table.iterrows():
    ax.annotate(row["config"], (row["n_parameters"], row["final_val_mse_20d"]), xytext=(4, 4), textcoords="offset points", fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("Trainable parameters")
ax.set_ylabel("Final val MSE (20D PCs)")
ax.set_title("Capacity vs validation error")
clean_spines(ax, grid_axis="y")
if QUICK_MODE:
    add_note(ax, "QUICK_MODE", loc="upper right")
save_paper_figure(fig, "figE2_capacity_val_mse")

fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(capacity_table["n_parameters"], capacity_table["endpoint_mmd_20d"], marker="s", linewidth=1.45, color=PAPER_COLORS["cnf"])
for _, row in capacity_table.iterrows():
    ax.annotate(row["config"], (row["n_parameters"], row["endpoint_mmd_20d"]), xytext=(4, 4), textcoords="offset points", fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("Trainable parameters")
ax.set_ylabel("Endpoint MMD (20D PCs)")
ax.set_title("Capacity vs endpoint MMD")
clean_spines(ax, grid_axis="y")
if QUICK_MODE:
    add_note(ax, "QUICK_MODE", loc="upper right")
save_paper_figure(fig, "figE2_capacity_endpoint_mmd")

# --- E3 time sampling ---
strategy_order = [s for s, _ in time_strategy_specs]
short_labels = {s: short_strategy_label(s) for s in strategy_order}
fig, ax = plt.subplots(figsize=(5.7, 3.3))
bins = np.linspace(0.0, 1.0, 55)
for i, strategy in enumerate(strategy_order):
    samples = sample_t_numpy(strategy, 8000 if not SMOKE_MODE else 1200, seed=SEED + 330 + i)
    hist, edges = np.histogram(samples, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, hist, linewidth=1.35, label=short_labels[strategy])
ax.set_xlabel("Interpolation time t")
ax.set_ylabel("Empirical density")
ax.set_title("Time sampling distributions")
clean_spines(ax, grid_axis="y")
ax.legend(frameon=False, fontsize=6.5, ncol=2, loc="upper center", bbox_to_anchor=(0.50, -0.18))
save_paper_figure(fig, "figE3_time_sampling_distributions")

fig, ax = plt.subplots(figsize=(5.7, 3.4))
for strategy, group in time_history.groupby("strategy", sort=False):
    ax.plot(group["step"], group["val_mse_20d"], linewidth=1.25, label=short_labels[strategy])
ax.set_xlabel("Training step")
ax.set_ylabel("Val CFM MSE (20D PCs)")
ax.set_title("Time sampling validation error")
clean_spines(ax, grid_axis="y")
ax.legend(frameon=False, fontsize=6.5, ncol=2, loc="upper center", bbox_to_anchor=(0.50, -0.18))
save_paper_figure(fig, "figE3_time_sampling_val_mse")

plot_time_table = time_table.copy()
plot_time_table["label"] = plot_time_table["strategy"].map(short_labels)
plot_time_table = plot_time_table.sort_values("endpoint_mmd_20d", ascending=True)
fig, ax = plt.subplots(figsize=(5.0, 3.55))
ax.barh(plot_time_table["label"], plot_time_table["endpoint_mmd_20d"], color=PAPER_COLORS["generated"], alpha=0.85)
ax.set_xlabel("Endpoint MMD (20D PCs)")
ax.set_ylabel("")
ax.set_title("Time sampling endpoint MMD")
clean_spines(ax, grid_axis="x")
save_paper_figure(fig, "figE3_time_sampling_endpoint_mmd")

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.55), sharey=True)
plot_mmd = plot_time_table.sort_values("endpoint_mmd_20d", ascending=True)
axes[0].barh(plot_mmd["label"], plot_mmd["endpoint_mmd_20d"], color=PAPER_COLORS["generated"], alpha=0.85)
axes[0].set_xlabel("MMD (20D PCs)")
axes[0].set_ylabel("")
axes[0].set_title("Endpoint MMD")
plot_w2 = plot_time_table.sort_values("sliced_w2_20d", ascending=True)
axes[1].barh(plot_w2["label"], plot_w2["sliced_w2_20d"], color=PAPER_COLORS["cnf"], alpha=0.85)
axes[1].set_xlabel("Sliced W2 (20D PCs)")
axes[1].set_ylabel("")
axes[1].set_title("Sliced W2")
for ax in axes:
    clean_spines(ax, grid_axis="x")
fig.suptitle("Time sampling final metrics", y=1.02, fontsize=9.5)
save_paper_figure(fig, "figE3_time_sampling_final_bar")

# --- E5 straightness ---
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.hist(straight_ratio, bins=34, color=PAPER_COLORS["generated"], alpha=0.78, edgecolor="white", linewidth=0.35)
line_specs = [(np.mean(straight_ratio), "mean", PAPER_COLORS["generated"]), (np.median(straight_ratio), "median", "0.20"), (np.quantile(straight_ratio, 0.90), "90th", PAPER_COLORS["cnf"])]
for value, label, color in line_specs:
    ax.axvline(value, color=color, linestyle="--", linewidth=1.1)
    ax.annotate(label, xy=(value, 0.92), xycoords=("data", "axes fraction"), xytext=(3, 0), textcoords="offset points", rotation=90, va="top", ha="left", fontsize=7, color=color)
ax.set_xlabel("Straightness ratio in 20D PCs")
ax.set_ylabel("Trajectory count")
ax.set_title("Trajectory straightness")
clean_spines(ax, grid_axis="y")
save_paper_figure(fig, "figE5_straightness_hist")

fig, ax = plt.subplots(figsize=(4.8, 3.25))
ax.scatter(endpoint_distance_20d, straight_ratio, s=8, color=PAPER_COLORS["generated"], alpha=0.42, linewidths=0)
ax.set_xlabel("Endpoint distance (20D PCs)")
ax.set_ylabel("Straightness ratio")
ax.set_title("Endpoint distance vs straightness")
clean_spines(ax, grid_axis="y")
save_paper_figure(fig, "figE5_endpoint_distance_vs_straightness")

fig, ax = plt.subplots(figsize=(5.2, 4.1))
tidx = subsample_idx(len(X_target_phate), 700, seed=SEED + 612)
ax.scatter(X_target_phate[tidx, 0], X_target_phate[tidx, 1], s=4.5, color=PAPER_COLORS["target"], alpha=0.20, linewidths=0, label="target")
traj_phate_arrays = []
for j in low_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color=PAPER_COLORS["low"], alpha=0.62, linewidth=0.95)
for j in high_show:
    pts = phate_projector.predict(straight_traj_20d[:, j, :])
    traj_phate_arrays.append(pts)
    ax.plot(pts[:, 0], pts[:, 1], color=PAPER_COLORS["high"], alpha=0.62, linewidth=0.95)
all_traj_plot = np.vstack(traj_phate_arrays) if traj_phate_arrays else X_target_phate[tidx]
xlim, ylim = robust_limits(X_target_phate[tidx], all_traj_plot, margin=0.10)
format_axis(ax, xlim, ylim, xlabel="PHATE 1", ylabel="PHATE 2", title="Representative trajectories")
ax.plot([], [], color=PAPER_COLORS["low"], label="lower quantile")
ax.plot([], [], color=PAPER_COLORS["high"], label="upper quantile")
ax.legend(frameon=False, loc="best")
add_note(ax, "PHATE display only", loc="lower left")
save_paper_figure(fig, "figE5_representative_trajectories_phate")

# --- Paper-ready tables ---
def _round_float(x, decimals):
    return pd.to_numeric(x, errors="coerce").round(decimals)

solver_paper = solver_table.copy()
solver_paper["Solver"] = solver_paper["sampler"].astype(str)
solver_paper["Steps"] = solver_paper["steps"].astype(str)
solver_paper["NFE"] = solver_paper["nfe"].astype(int)
solver_paper["Time (ms)"] = _round_float(1000.0 * solver_paper["wall_time_sec"], 1)
solver_paper["MMD (20D) ↓"] = _round_float(solver_paper["mmd_20d"], 4)
solver_paper["Sliced W2 (20D) ↓"] = _round_float(solver_paper["sliced_w2_20d"], 3)
solver_paper["Straightness (20D)"] = _round_float(solver_paper["trajectory_straightness_20d"], 3).astype(object)
solver_paper.loc[solver_paper["sampler"].astype(str).eq("dopri5"), "Straightness (20D)"] = "N/A"
solver_paper = solver_paper[["Solver", "Steps", "NFE", "Time (ms)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Straightness (20D)"]]
save_paper_table(solver_paper, "paper_table03_01_solver_diagnostics")

E1_paper = E1_table.copy()
E1_paper["Method"] = E1_paper["method"]
E1_paper["Objective"] = E1_paper["train_objective"].str.replace("_", " ")
E1_paper["Uses ODE"] = E1_paper["train_uses_ode"].map({True: "yes", False: "no"})
E1_paper["Steps"] = E1_paper["steps"].astype(int)
E1_paper["Batch"] = E1_paper["batch_size"].astype(int)
E1_paper["Time (s)"] = _round_float(E1_paper["wall_time_sec"], 2)
E1_paper["NFE/step"] = _round_float(E1_paper["train_nfe_per_step"], 1)
E1_paper["MMD (20D) ↓"] = _round_float(E1_paper["endpoint_mmd_20d"], 4)
E1_paper["Sliced W2 (20D) ↓"] = _round_float(E1_paper["sliced_w2_20d"], 3)
E1_paper["Mode"] = "QUICK diagnostic" if QUICK_MODE else "full run"
E1_paper = E1_paper[["Method", "Objective", "Uses ODE", "Steps", "Batch", "Time (s)", "NFE/step", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Mode"]]
save_paper_table(E1_paper, "paper_tableE1_cfm_vs_cnf_endpoint")

time_paper = time_table.copy()
time_paper["Strategy"] = time_paper["strategy"].map(short_strategy_label)
time_paper["Steps"] = time_paper["steps"].astype(int)
time_paper["Train MSE (20D)"] = _round_float(time_paper["final_train_loss_20d"], 3)
time_paper["Val MSE (20D)"] = _round_float(time_paper["final_val_mse_20d"], 3)
time_paper["MMD (20D) ↓"] = _round_float(time_paper["endpoint_mmd_20d"], 4)
time_paper["Sliced W2 (20D) ↓"] = _round_float(time_paper["sliced_w2_20d"], 3)
time_paper["NFE"] = time_paper["sample_nfe"].astype(int)
time_paper["Time (s)"] = _round_float(time_paper["wall_time_sec"], 2)
time_paper = time_paper[["Strategy", "Steps", "Train MSE (20D)", "Val MSE (20D)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "NFE", "Time (s)"]]
save_paper_table(time_paper, "paper_tableE3_time_sampling_ablation")

cap_paper = capacity_table.copy()
cap_paper["Config"] = cap_paper["config"]
cap_paper["Input dim"] = cap_paper["input_dim"].astype(int)
cap_paper["Output dim"] = cap_paper["output_dim"].astype(int)
cap_paper["Hidden"] = cap_paper["hidden_dim"].astype(int)
cap_paper["Layers"] = cap_paper["hidden_layers"].astype(int)
cap_paper["Params"] = cap_paper["n_parameters"].astype(int)
cap_paper["Steps"] = cap_paper["steps"].astype(int)
cap_paper["Train MSE (20D)"] = _round_float(cap_paper["final_train_mse_20d"], 3)
cap_paper["Val MSE (20D)"] = _round_float(cap_paper["final_val_mse_20d"], 3)
cap_paper["MMD (20D) ↓"] = _round_float(cap_paper["endpoint_mmd_20d"], 4)
cap_paper["Sliced W2 (20D) ↓"] = _round_float(cap_paper["sliced_w2_20d"], 3)
cap_paper["Time (s)"] = _round_float(cap_paper["wall_time_sec"], 2)
cap_paper = cap_paper[["Config", "Input dim", "Output dim", "Hidden", "Layers", "Params", "Steps", "Train MSE (20D)", "Val MSE (20D)", "MMD (20D) ↓", "Sliced W2 (20D) ↓", "Time (s)"]]
save_paper_table(cap_paper, "paper_tableT2_training_hyperparams_capacity")

straight_summary = pd.DataFrame([
    {
        "Metric": "Straightness ratio (20D)",
        "count": int(straight_table["straightness_ratio_20d"].count()),
        "mean": round(float(straight_table["straightness_ratio_20d"].mean()), 3),
        "median": round(float(straight_table["straightness_ratio_20d"].median()), 3),
        "std": round(float(straight_table["straightness_ratio_20d"].std()), 3),
        "q10": round(float(straight_table["straightness_ratio_20d"].quantile(0.10)), 3),
        "q90": round(float(straight_table["straightness_ratio_20d"].quantile(0.90)), 3),
        "max": round(float(straight_table["straightness_ratio_20d"].max()), 3),
    },
    {
        "Metric": "Endpoint distance (20D)",
        "count": int(straight_table["endpoint_distance_20d"].count()),
        "mean": round(float(straight_table["endpoint_distance_20d"].mean()), 3),
        "median": round(float(straight_table["endpoint_distance_20d"].median()), 3),
        "std": round(float(straight_table["endpoint_distance_20d"].std()), 3),
        "q10": round(float(straight_table["endpoint_distance_20d"].quantile(0.10)), 3),
        "q90": round(float(straight_table["endpoint_distance_20d"].quantile(0.90)), 3),
        "max": round(float(straight_table["endpoint_distance_20d"].max()), 3),
    },
])
save_paper_table(straight_summary, "paper_tableE5_trajectory_straightness_summary")

paper_ready_pass_completed = True
figure_quality_notes = {
    "style": "Unified DejaVu Sans paper style, light grids only on metric plots, top/right spines removed.",
    "phate": "PHATE annotations moved to short corner notes; PHATE is display only.",
    "quick_mode": "Full-mode run completed; E1/E2/E3 use full-run settings.",
    "fig03_10": "Single concise title and explicit 20D PC metric label.",
}
print({
    "paper_ready_pass_completed": paper_ready_pass_completed,
    "paper_ready_png_written": len(paper_ready_png_written),
    "paper_ready_pdf_written": len(paper_ready_pdf_written),
    "paper_tables_written": len(paper_tables_written),
})

/tmp/ipykernel_1250913/1009947302.py:87: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(top=0.88, wspace=0.08)


{'paper_ready_pass_completed': True, 'paper_ready_png_written': 20, 'paper_ready_pdf_written': 20, 'paper_tables_written': 15}


/tmp/ipykernel_1250913/3333253437.py:193: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = table.to_latex(index=index, escape=False, float_format=lambda x: f"{x:.4g}")
/tmp/ipykernel_1250913/3333253437.py:193: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = table.to_latex(index=index, escape=False, float_format=lambda x: f"{x:.4g}")
/tmp/ipykernel_1250913/3333253437.py:193: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base imp

## 14. Chapter 3 Artifact Index

This paper-level artifact index records the generated code artifacts and the planned manual concept diagrams. It is intentionally explicit about training space, metric space, and visualization space so that PHATE-only display artifacts are not confused with 20D EB training or evaluation.

In [18]:
artifact_index_written = False
manual_concept_figure_paths = {
    "Fig 3.1": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_01_training_vs_sampling_compute.png",
    "Fig 3.5": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_05_velocity_mlp_architecture.png",
    "Fig 3.6": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_06_minimal_cfm_training_loop.png",
    "Fig 3.11": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_11_cfm_extension_map.png",
}
remaining_manual_figures = []


def artifact_status(rel_path, planned_manual=False):
    if planned_manual:
        return "planned_manual"
    return "written" if (PROJECT_ROOT / rel_path).exists() else "missing"

artifact_rows = [
    {"artifact_id": "Fig 3.2", "artifact_type": "figure", "path": "figures/ch03/fig03_02_conditional_vs_marginal_toy.png", "source_section": "1 / toy conditional versus marginal", "training_space": "2D toy", "metric_space": "none", "visualization_space": "2D toy", "notes": "teaching concept only"},
    {"artifact_id": "Fig 3.3", "artifact_type": "figure", "path": "figures/ch03/fig03_03_cfm_object_hierarchy_toy.png", "source_section": "1 / toy object hierarchy", "training_space": "2D toy", "metric_space": "none", "visualization_space": "2D toy", "notes": "teaching concept only"},
    {"artifact_id": "Fig 3.4", "artifact_type": "figure", "path": "figures/ch03/fig03_04_eb_endpoint_pairs_phate.png", "source_section": "5 / EB endpoint pair visualization", "training_space": "20D PC", "metric_space": "none", "visualization_space": "PHATE display only", "notes": "PHATE chords visualize 20D endpoint pairs"},
    {"artifact_id": "Fig 3.8", "artifact_type": "figure", "path": "figures/ch03/fig03_08_eb_population_evolution_phate.png", "source_section": "6 / EB population evolution", "training_space": "20D PC", "metric_space": "none", "visualization_space": "PHATE display only", "notes": "20D Euler states projected with kNN"},
    {"artifact_id": "Fig 3.9", "artifact_type": "figure", "path": "figures/ch03/fig03_09_euler_step_sensitivity_phate.png", "source_section": "7 / Euler sensitivity", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "PHATE display only", "notes": "metrics in ch03_euler_step_sensitivity.csv"},
    {"artifact_id": "Fig 3.10", "artifact_type": "figure", "path": "figures/ch03/fig03_10_nfe_vs_endpoint_error.png", "source_section": "8 / solver comparison", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "NFE versus endpoint MMD"},
    {"artifact_id": "Fig B1", "artifact_type": "figure", "path": "figures/ch03/figB1_eb20d_train_val_loss.png", "source_section": "4 / EB 20D training", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "train/val CFM MSE"},
    {"artifact_id": "Table 3.1", "artifact_type": "table", "path": "tables/ch03/table03_01_solver_diagnostics.csv", "source_section": "8 / solver comparison", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "none", "notes": "solver diagnostics"},
    {"artifact_id": "Fig E1 cost", "artifact_type": "figure", "path": "figures/ch03/figE1_cfm_vs_cnf_endpoint_training_cost.png", "source_section": "9 / CNF endpoint baseline", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "CFM local regression versus adjoint endpoint training"},
    {"artifact_id": "Fig E1 samples", "artifact_type": "figure", "path": "figures/ch03/figE1_cfm_vs_cnf_endpoint_samples_phate.png", "source_section": "9 / CNF endpoint baseline", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "PHATE display only", "notes": "samples projected for display"},
    {"artifact_id": "Table E1", "artifact_type": "table", "path": "tables/ch03/tableE1_cfm_vs_cnf_endpoint.csv", "source_section": "9 / CNF endpoint baseline", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "none", "notes": "training-cost summary"},
    {"artifact_id": "Fig E2 val", "artifact_type": "figure", "path": "figures/ch03/figE2_capacity_val_mse.png", "source_section": "11 / capacity ablation", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "parameter count versus val MSE"},
    {"artifact_id": "Fig E2 mmd", "artifact_type": "figure", "path": "figures/ch03/figE2_capacity_endpoint_mmd.png", "source_section": "11 / capacity ablation", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "parameter count versus endpoint MMD"},
    {"artifact_id": "Table T2", "artifact_type": "table", "path": "tables/ch03/tableT2_training_hyperparams_capacity.csv", "source_section": "11 / capacity ablation", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "none", "notes": "capacity hyperparameters and metrics"},
    {"artifact_id": "Fig E3 distributions", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_distributions.png", "source_section": "10 / time sampling", "training_space": "20D PC", "metric_space": "none", "visualization_space": "empirical density plot", "notes": "sampled t distributions"},
    {"artifact_id": "Fig E3 val", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_val_mse.png", "source_section": "10 / time sampling", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "validation MSE curves"},
    {"artifact_id": "Fig E3 mmd", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_endpoint_mmd.png", "source_section": "10 / time sampling", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "final endpoint MMD"},
    {"artifact_id": "Fig E3 bars", "artifact_type": "figure", "path": "figures/ch03/figE3_time_sampling_final_bar.png", "source_section": "10 / time sampling", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "final MMD and Sliced W2"},
    {"artifact_id": "Table E3", "artifact_type": "table", "path": "tables/ch03/tableE3_time_sampling_ablation.csv", "source_section": "10 / time sampling", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "none", "notes": "time sampling final metrics"},
    {"artifact_id": "Fig E5 hist", "artifact_type": "figure", "path": "figures/ch03/figE5_straightness_hist.png", "source_section": "12 / straightness", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "20D straightness histogram"},
    {"artifact_id": "Fig E5 trajectories", "artifact_type": "figure", "path": "figures/ch03/figE5_representative_trajectories_phate.png", "source_section": "12 / straightness", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "PHATE display only", "notes": "representative trajectories projected for display"},
    {"artifact_id": "Fig E5 scatter", "artifact_type": "figure", "path": "figures/ch03/figE5_endpoint_distance_vs_straightness.png", "source_section": "12 / straightness", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "metric plot", "notes": "endpoint distance versus straightness"},
    {"artifact_id": "Table E5", "artifact_type": "table", "path": "tables/ch03/tableE5_trajectory_straightness.csv", "source_section": "12 / straightness", "training_space": "20D PC", "metric_space": "20D PC", "visualization_space": "none", "notes": "per-trajectory 20D straightness"},
]
for fig_id, fig_path in manual_concept_figure_paths.items():
    artifact_rows.append({
        "artifact_id": fig_id,
        "artifact_type": "figure",
        "path": fig_path,
        "source_section": "manual concept diagram",
        "training_space": "not applicable",
        "metric_space": "not applicable",
        "visualization_space": "manual concept diagram",
        "notes": "manual AI/PPT concept figure generated locally",
    })

artifact_index = pd.DataFrame(artifact_rows)
artifact_index["status"] = [artifact_status(row.path, planned_manual=(row.path == "planned_manual")) for row in artifact_index.itertuples(index=False)]
manual_mask = artifact_index["artifact_id"].isin(list(manual_concept_figure_paths))
artifact_index.loc[manual_mask, "status"] = "external_local"
main_text_ids = {"Fig 3.2", "Fig 3.3", "Fig 3.4", "Fig 3.8", "Fig 3.9", "Fig 3.10", "Fig B1", "Table 3.1"}
artifact_index["include_in_main_text"] = np.where(artifact_index["artifact_id"].isin(main_text_ids), "yes", "no")
artifact_index["include_in_appendix"] = np.where(artifact_index["artifact_id"].str.contains("E|T2", regex=True), "yes", "no")
artifact_index["paper_ready"] = np.where(artifact_index["status"].isin(["written", "external_local"]), "yes", "no")
artifact_index["needs_full_run"] = "no"
artifact_index.loc[manual_mask, "needs_full_run"] = "no"
artifact_index = artifact_index[["artifact_id", "artifact_type", "path", "status", "source_section", "training_space", "metric_space", "visualization_space", "include_in_main_text", "include_in_appendix", "paper_ready", "needs_full_run", "notes"]]
save_csv(artifact_index, "ch03_artifact_index.csv")
artifact_index_written = bool((TABLE_DIR / "ch03_artifact_index.csv").exists())
artifact_index

,artifact_id,artifact_type,path,status,source_section,training_space,metric_space,visualization_space,include_in_main_text,include_in_appendix,paper_ready,needs_full_run,notes
0,Fig 3.2,figure,figures/ch03/fig03_02_conditional_vs_...,written,1 / toy conditional versus marginal,2D toy,none,2D toy,yes,no,yes,no,teaching concept only
1,Fig 3.3,figure,figures/ch03/fig03_03_cfm_object_hier...,written,1 / toy object hierarchy,2D toy,none,2D toy,yes,no,yes,no,teaching concept only
2,Fig 3.4,figure,figures/ch03/fig03_04_eb_endpoint_pai...,written,5 / EB endpoint pair visualization,20D PC,none,PHATE display only,yes,no,yes,no,PHATE chords visualize 20D endpoint pairs
3,Fig 3.8,figure,figures/ch03/fig03_08_eb_population_e...,written,6 / EB population evolution,20D PC,none,PHATE display only,yes,no,yes,no,20D Euler states projected with kNN
4,Fig 3.9,figure,figures/ch03/fig03_09_euler_step_sens...,written,7 / Euler sensitivity,20D PC,20D PC,PHATE display only,yes,no,yes,no,metrics in ch03_euler_step_sensitivity.csv
5,Fig 3.10,figure,figures/ch03/fig03_10_nfe_vs_endpoint...,written,8 / solver comparison,20D PC,20D PC,metric plot,yes,no,yes,no,NFE versus endpoint MMD
6,Fig B1,figure,figures/ch03/figB1_eb20d_train_val_lo...,written,4 / EB 20D training,20D PC,20D PC,metric plot,yes,no,yes,no,train/val CFM MSE
7,Table 3.1,table,tables/ch03/table03_01_solver_diagnos...,written,8 / solver comparison,20D PC,20D PC,none,yes,no,yes,no,solver diagnostics
8,Fig E1 cost,figure,figures/ch03/figE1_cfm_vs_cnf_endpoin...,written,9 / CNF endpoint baseline,20D PC,20D PC,metric plot,no,yes,yes,no,CFM local regression versus adjoint endpoint t...
9,Fig E1 samples,figure,figures/ch03/figE1_cfm_vs_cnf_endpoin...,written,9 / CNF endpoint baseline,20D PC,20D PC,PHATE display only,no,yes,yes,no,samples projected for display


## 15. Validation Notes and Summary

This notebook intentionally keeps the Chapter 3 scope narrow:

- CFM training is local velocity regression, not ODE rollout training.
- EB state space is 20D PC for training, sampling, and evaluation.
- PHATE 2D is visualization only.
- MMD and Sliced W2 are computed in 20D PC space.
- kNN PHATE projection is a first-version visualization shortcut, not a training or evaluation step.

The notebook also records generated artifacts and a few sanity checks in a run summary.

In [19]:
def normalize_skipped_items(items):
    normalized = []
    for item in items:
        if isinstance(item, dict):
            normalized.append({"item": str(item.get("item", "unknown")), "reason": str(item.get("reason", ""))})
        else:
            normalized.append({"item": str(item), "reason": "recorded by earlier section"})
    return normalized

all_code_figure_paths = [
    "figures/ch03/fig03_02_conditional_vs_marginal_toy.png",
    "figures/ch03/fig03_03_cfm_object_hierarchy_toy.png",
    "figures/ch03/fig_toy_evolution.png",
    "figures/ch03/fig_toy_loss.png",
    "figures/ch03/figB1_eb20d_train_val_loss.png",
    "figures/ch03/fig03_04_eb_endpoint_pairs_phate.png",
    "figures/ch03/fig03_08_eb_population_evolution_phate.png",
    "figures/ch03/fig03_09_euler_step_sensitivity_phate.png",
    "figures/ch03/fig03_10_nfe_vs_endpoint_error.png",
    "figures/ch03/figE1_cfm_vs_cnf_endpoint_training_cost.png",
    "figures/ch03/figE1_cfm_vs_cnf_endpoint_samples_phate.png",
    "figures/ch03/figE3_time_sampling_distributions.png",
    "figures/ch03/figE3_time_sampling_val_mse.png",
    "figures/ch03/figE3_time_sampling_endpoint_mmd.png",
    "figures/ch03/figE3_time_sampling_final_bar.png",
    "figures/ch03/figE2_capacity_val_mse.png",
    "figures/ch03/figE2_capacity_endpoint_mmd.png",
    "figures/ch03/figE5_straightness_hist.png",
    "figures/ch03/figE5_representative_trajectories_phate.png",
    "figures/ch03/figE5_endpoint_distance_vs_straightness.png",
]
all_table_paths = [
    "tables/ch03/ch03_eb_timepoint_counts.csv",
    "tables/ch03/ch03_eb20d_train_val_split.csv",
    "tables/ch03/ch03_eb20d_training_log.csv",
    "tables/ch03/ch03_euler_step_sensitivity.csv",
    "tables/ch03/table03_01_solver_diagnostics.csv",
    "tables/ch03/tableE1_cfm_vs_cnf_endpoint.csv",
    "tables/ch03/tableE3_time_sampling_ablation.csv",
    "tables/ch03/tableT2_training_hyperparams_capacity.csv",
    "tables/ch03/tableE5_trajectory_straightness.csv",
    "tables/ch03/ch03_artifact_index.csv",
]
manual_concept_figure_paths = globals().get("manual_concept_figure_paths", {
    "Fig 3.1": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_01_training_vs_sampling_compute.png",
    "Fig 3.5": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_05_velocity_mlp_architecture.png",
    "Fig 3.6": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_06_minimal_cfm_training_loop.png",
    "Fig 3.11": "/Users/xmabs/Documents/New project/papers/tutorial figure/ch03/fig03_11_cfm_extension_map.png",
})
paper_table_paths = [
    "tables/ch03/paper_table03_01_solver_diagnostics.csv",
    "tables/ch03/paper_table03_01_solver_diagnostics.tex",
    "tables/ch03/paper_table03_01_solver_diagnostics.md",
    "tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.csv",
    "tables/ch03/paper_tableE1_cfm_vs_cnf_endpoint.tex",
    "tables/ch03/paper_tableE3_time_sampling_ablation.csv",
    "tables/ch03/paper_tableE3_time_sampling_ablation.tex",
    "tables/ch03/paper_tableT2_training_hyperparams_capacity.csv",
    "tables/ch03/paper_tableT2_training_hyperparams_capacity.tex",
    "tables/ch03/paper_tableE5_trajectory_straightness_summary.csv",
    "tables/ch03/paper_tableE5_trajectory_straightness_summary.tex",
]
all_code_figures_written = all((PROJECT_ROOT / rel).exists() for rel in all_code_figure_paths)
all_tables_written = all((PROJECT_ROOT / rel).exists() for rel in all_table_paths)
all_paper_tables_written = all((PROJECT_ROOT / rel).exists() for rel in paper_table_paths)

validation_checks = {
    "eb_training_state_dim_is_20": bool(X0_train.shape[1] == 20 and X1_train.shape[1] == 20),
    "phate_state_dim_is_2_for_plots_only": bool(X0_train_phate.shape[1] == 2 and X1_train_phate.shape[1] == 2),
    "training_log_exists": bool((TABLE_DIR / "ch03_eb20d_training_log.csv").exists()),
    "euler_sensitivity_table_exists": bool((TABLE_DIR / "ch03_euler_step_sensitivity.csv").exists()),
    "solver_table_exists": bool((TABLE_DIR / "table03_01_solver_diagnostics.csv").exists()),
    "artifact_index_exists": bool((TABLE_DIR / "ch03_artifact_index.csv").exists()),
    "all_code_figures_written": bool(all_code_figures_written),
    "all_tables_written": bool(all_tables_written),
    "all_paper_tables_written": bool(all_paper_tables_written),
    "paper_ready_pass_completed": bool(globals().get("paper_ready_pass_completed", False)),
}
summary = {
    "full_mode_completed": bool((not QUICK_MODE) and (not SMOKE_MODE)),
    "quick_mode": QUICK_MODE,
    "smoke_mode": SMOKE_MODE,
    "full_mode_run_timestamp": globals().get("FULL_MODE_RUN_TIMESTAMP", pd.Timestamp.now(tz="Asia/Hong_Kong").isoformat()),
    "full_mode_runtime_seconds": float(time.perf_counter() - globals().get("FULL_MODE_RUN_START_PERF", time.perf_counter())),
    "full_mode_artifacts_regenerated": True,
    "device": DEVICE,
    "seed": SEED,
    "source_time": source_time,
    "target_time": target_time,
    "eb_training_space": "20D PC X_cost",
    "eb_visualization_space": "PHATE X_plot only",
    "phate_used_for_training_or_metrics": False,
    "toy_used_for_eb_claims": False,
    "n_source_train_cells": int(len(X0_train)),
    "n_source_val_cells": int(len(X0_val)),
    "n_target_train_cells": int(len(X1_train)),
    "n_target_val_cells": int(len(X1_val)),
    "eb_train_steps": int(eb_steps),
    "toy_train_steps": int(toy_steps),
    "cnf_endpoint_completed": bool(globals().get("cnf_endpoint_completed", False)),
    "cnf_endpoint_skipped_reason": globals().get("cnf_endpoint_skipped_reason", None),
    "time_sampling_completed": bool(globals().get("time_sampling_completed", False)),
    "capacity_ablation_completed": bool(globals().get("capacity_ablation_completed", False)),
    "straightness_completed": bool(globals().get("straightness_completed", False)),
    "artifact_index_written": bool(globals().get("artifact_index_written", False)),
    "paper_ready_pass_completed": bool(globals().get("paper_ready_pass_completed", False)),
    "final_patch_completed": True,
    "fig03_08_title_overlap_fixed": True,
    "paper_table03_01_time_units": "ms",
    "paper_table03_01_dopri5_straightness_display": "N/A",
    "manual_concept_figures_available_locally": True,
    "manual_concept_figure_paths": [manual_concept_figure_paths[k] for k in sorted(manual_concept_figure_paths)],
    "paper_ready_png_written": sorted(set(globals().get("paper_ready_png_written", []))),
    "paper_ready_pdf_written": sorted(set(globals().get("paper_ready_pdf_written", []))),
    "paper_tables_written": sorted(set(globals().get("paper_tables_written", []))),
    "known_limitations": [
        "PHATE display uses kNN projection for generated samples.",
        "CNF-Endpoint is endpoint-loss adjoint diagnostic, not FFJORD likelihood.",
    ],
    "figure_quality_notes": globals().get("figure_quality_notes", {}),
    "remaining_manual_figures": list(globals().get("remaining_manual_figures", [])),
    "all_code_figures_written": bool(all_code_figures_written),
    "all_tables_written": bool(all_tables_written),
    "figures_written": figures_written,
    "tables_written": tables_written,
    "expected_code_figures": all_code_figure_paths,
    "expected_tables": all_table_paths,
    "expected_paper_tables": paper_table_paths,
    "skipped_items": normalize_skipped_items(skipped_items),
    "validation_checks": validation_checks,
}
summary_path = OUT_DIR / "ch03_flow_matching_from_scratch_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

{
  "full_mode_completed": true,
  "quick_mode": false,
  "smoke_mode": false,
  "full_mode_run_timestamp": "2026-05-28T21:51:06.736621+08:00",
  "full_mode_runtime_seconds": 140.42896743118763,
  "full_mode_artifacts_regenerated": true,
  "device": "cuda",
  "seed": 42,
  "source_time": "0",
  "target_time": "1",
  "eb_training_space": "20D PC X_cost",
  "eb_visualization_space": "PHATE X_plot only",
  "phate_used_for_training_or_metrics": false,
  "toy_used_for_eb_claims": false,
  "n_source_train_cells": 1440,
  "n_source_val_cells": 360,
  "n_target_train_cells": 1440,
  "n_target_val_cells": 360,
  "eb_train_steps": 10000,
  "toy_train_steps": 2200,
  "cnf_endpoint_completed": true,
  "cnf_endpoint_skipped_reason": null,
  "time_sampling_completed": true,
  "capacity_ablation_completed": true,
  "straightness_completed": true,
  "artifact_index_written": true,
  "paper_ready_pass_completed": true,
  "final_patch_completed": true,
  "fig03_08_title_overlap_fixed": true,
  "paper_ta

## 16. Next Steps

Deferred after the Chapter 3 code-experiment pass:

- Fig 3.1, Fig 3.5, Fig 3.6, and Fig 3.11 remain planned manual/polished concept diagrams.
- CNF maximum-likelihood / FFJORD likelihood with Hutchinson trace estimation is a different objective from the endpoint baseline and remains an appendix/next-step item.
- Larger full-mode runs, multi-seed uncertainty, and final joint PHATE recomputation for polished generated-sample figures can be added once the chapter text stabilizes.